In [ ]:
# Solar Filament Segmentation V1.0 — single code cell
# Default: prepare -> two-GPU detector/refiner training -> official PQ calibration -> submission.csv.
# Kaggle: add competition data, select GPU T4 x2, enable Internet for official COCO/ImageNet weights.
# Offline: attach the official weight files, set PRETRAINED_DIR and ALLOW_WEIGHT_DOWNLOAD=False.
# This cell does not download competition data. All original images/labels stay on Kaggle.
# Optional Tile U-Net is fully implemented and disabled by default.
# RUN_MODE: train_calibrate_submit / train / prepare / validate / calibrate / predict.
# RESUME=True resumes matching last checkpoints. Keep OUTPUT_ROOT for the same experiment only.
# Predict-only or resume: attach a completed V1 output and set RESUME_SOURCE to that folder.
# RUN_MODE='predict' loads its saved calibration and weights without training.
from pathlib import Path
import os, sys, json, subprocess
assert Path('/kaggle/input').is_dir(), 'Run on Kaggle. Do not download competition data to this PC.'


CONFIG = {'RUN_MODE': 'train_calibrate_submit',
 'INPUT_ROOT': '/kaggle/input',
 'OUTPUT_ROOT': '/kaggle/working/solar_v1',
 'ANNOTATION_PATH': '',
 'TRAIN_IMAGE_DIR': '',
 'TEST_IMAGE_DIR': '',
 'SEED': 2026,
 'N_FOLDS': 5,
 'FOLD_INDEX': 0,
 'INPUT_MODE': 'raw_corrected_radial',
 'ANNOTATION_MODE': 'all_annotators',
 'PREPROCESS': True,
 'DISK_MARGIN': 1.01,
 'DETECTOR_IMAGE_SIZE': 1024,
 'ANCHOR_SIZES': (16, 32, 64, 128, 256),
 'ANCHOR_RATIOS': (0.5, 1.0, 2.0),
 'DETECTOR_EPOCHS': 10,
 'TRAIN_DETECTOR': True,
 'DETECTOR_LR': 0.005,
 'DETECTOR_ACCUM_STEPS': 1,
 'ENABLE_REFINER': True,
 'REFINER_SIZE': 384,
 'REFINER_EPOCHS': 8,
 'TRAIN_REFINER': True,
 'REFINER_BATCH_SIZE': 4,
 'REFINER_ACCUM_STEPS': 2,
 'REFINER_USE_COARSE': True,
 'REFINER_USE_RADIAL': True,
 'REFINER_CONTEXT': 1.8,
 'REFINER_MIN_CROP': 96,
 'REFINER_CROPS_PER_PHOTO': 8,
 'REFINER_BOX_JITTER': 0.12,
 'EMPTY_REFINER_FALLBACK': True,
 'SEGMENTER_LR': 0.0003,
 'TVERSKY_ALPHA': 0.3,
 'TVERSKY_BETA': 0.7,
 'ENABLE_TILE_BRANCH': False,
 'TILE_SIZE': 512,
 'TILE_OVERLAP': 128,
 'TILE_EPOCHS': 8,
 'TRAIN_TILE': True,
 'TILE_BATCH_SIZE': 2,
 'TILE_ACCUM_STEPS': 2,
 'TILE_SAMPLES_PER_PHOTO': 4,
 'TILE_FUSION_WEIGHT': 0.35,
 'TILE_THRESHOLD': 0.65,
 'TILE_GROW': 8,
 'TILE_KEEP_UNCLAIMED': False,
 'TILE_UNCLAIMED_MIN_AREA': 64,
 'DETECTION_THRESHOLD': 0.5,
 'MASK_THRESHOLD': 0.5,
 'REFINER_THRESHOLD': 0.65,
 'MASK_NMS_IOU': 0.5,
 'MIN_INSTANCE_AREA': 32,
 'POSTPROCESS_MODE': 'mask_nms',
 'SOFT_NMS_SIGMA': 0.5,
 'MORPH_CLOSE_RADIUS': 0,
 'FRAGMENT_MODE': 'keep',
 'FRAGMENT_MIN_AREA': 8,
 'CACHE_SCORE_FLOOR': 0.2,
 'MAX_DETECTIONS': 80,
 'INFERENCE_BATCH_SIZE': 8,
 'NUM_WORKERS': 2,
 'USE_DDP': True,
 'AMP': True,
 'VALIDATE_EVERY': 2,
 'RESUME': True,
 'PRETRAINED': True,
 'ALLOW_WEIGHT_DOWNLOAD': True,
 'PRETRAINED_DIR': '',
 'CHECKPOINT_DIR': '',
 'RESUME_SOURCE': '',
 'REFIT_ALL_DATA': False,
 'CROSS_FIT_CALIBRATION': True,
 'CALIBRATION_ROUNDS': 2,
 'CALIBRATION_GRID': {'DETECTION_THRESHOLD': [0.3, 0.5, 0.7],
                      'MASK_THRESHOLD': [0.4, 0.5, 0.6],
                      'REFINER_THRESHOLD': [0.5, 0.65, 0.8],
                      'MASK_NMS_IOU': [0.3, 0.5, 0.7],
                      'MIN_INSTANCE_AREA': [16, 32, 64]},
 'ENSEMBLE_CACHE_DIRS': (),
 'ENSEMBLE_WEIGHTS': (),
 'ENSEMBLE_MATCH_IOU': 0.55,
 'QUALITY_WEIGHT': 0.0,
 'MAX_CACHE_GB': 12.0}


# Dependencies: use Kaggle's matching PyTorch/torchvision CUDA installation.
import importlib.util
if importlib.util.find_spec('pycocotools') is None:
    subprocess.run([sys.executable,'-m','pip','install','-q','pycocotools==2.0.11'],check=True)
PACKAGE_ROOT=Path('/kaggle/working/solar_v1_runtime')
PACKAGE=PACKAGE_ROOT/'solar_v1'
PACKAGE.mkdir(parents=True,exist_ok=True)


(PACKAGE / '__init__.py').write_text(r'''"""Solar Filament Segmentation V1.0."""
__version__ = '1.0.0'
''', encoding='utf-8')


(PACKAGE / 'calibration.py').write_text(r'''"""Date-blocked cross-fitting of end-to-end PQ thresholds, conditional on selected models."""
from dataclasses import asdict,replace
from pathlib import Path
import json
import numpy as np
import pandas as pd
from .config import atomic_json,digest,file_hash
from .evaluation import evaluate_cache


THRESHOLDS=['DETECTION_THRESHOLD','MASK_THRESHOLD','REFINER_THRESHOLD','MASK_NMS_IOU','MIN_INSTANCE_AREA']


def counts_pq(values):
    v=np.asarray(values,np.float64).sum(0);tp,fp,fn,iou=v
    return float(iou/(tp+.5*fp+.5*fn)) if tp+.5*fp+.5*fn else 0.


def crossfit_partition(photos,seed):
    groups=sorted({p.get('group',p['date']) for p in photos})
    if len(groups)<2:raise ValueError('Cross-fitting needs at least two independent validation date groups')
    np.random.default_rng(seed).shuffle(groups);left=set(groups[:len(groups)//2])
    a=np.array([i for i,p in enumerate(photos) if p.get('group',p['date']) in left],int)
    b=np.array([i for i,p in enumerate(photos) if p.get('group',p['date']) not in left],int)
    return a,b


def calibrate(c,cat,photos,cache,pre):
    out=Path(c.OUTPUT_ROOT)/'calibration';out.mkdir(exist_ok=True)
    all_values={};history=[]
    external_fingerprints={str(root):{name:file_hash(Path(root)/name) for name in
        ['ensemble_manifest.json']+[p['stem']+'.json' for p in photos]} for root in c.ENSEMBLE_CACHE_DIRS}
    def values(params):
        trial=replace(c,**params)
        key=digest({'config':asdict(trial),'cache':cache.signature,'photos':[p['stem'] for p in photos],
                    'external':external_fingerprints,'annotations':cat.hash})
        if key not in all_values:
            path=out/(key[:24]+'.json')
            if path.exists():payload=json.loads(path.read_text())
            else:
                ev=evaluate_cache(trial,cat,photos,cache,pre)
                payload={'params':params,'counts':[[ev.per_photo[p['stem']][f] for f in ['tp','fp','fn','iou_sum']] for p in photos]}
                atomic_json(path,payload)
            all_values[key]=np.asarray(payload['counts'],np.float64)
        return all_values[key]
    def search(indices,label):
        best={k:getattr(c,k) for k in THRESHOLDS};score=counts_pq(values(best)[indices])
        for round_no in range(c.CALIBRATION_ROUNDS):
            changed=False
            for name in THRESHOLDS:
                if name=='REFINER_THRESHOLD' and not c.ENABLE_REFINER:continue
                chosen=best;chosen_score=score
                for val in c.CALIBRATION_GRID[name]:
                    params={**best,name:val};pq=counts_pq(values(params)[indices])
                    history.append({'fit':label,'round':round_no,'parameter':name,'value':val,'pq':pq,**params})
                    if pq>chosen_score+1e-12:chosen=params;chosen_score=pq
                changed |= chosen!=best;best,score=chosen,chosen_score
            print(f'Calibration {label} round {round_no+1}: PQ={score:.6f} {best}',flush=True)
            if not changed:break
        return best,score
    all_idx=np.arange(len(photos));best,tuned=search(all_idx,'all_validation_for_deployment')
    report={'deployment_thresholds':best,'same_validation_tuned_pq':tuned,
            'method':'coordinate search; all five thresholds scored using the complete pipeline',
            'checkpoint_selection_note':'Models were selected using this validation set. Threshold cross-fitting alone is not full model OOF.'}
    if c.CROSS_FIT_CALIBRATION:
        a,b=crossfit_partition(photos,c.SEED)
        fit_a,_=search(a,'fit_A_score_B');fit_b,_=search(b,'fit_B_score_A')
        heldout=np.zeros((len(photos),4))
        heldout[b]=values(fit_a)[b];heldout[a]=values(fit_b)[a]
        report.update(cross_fitted_threshold_pq=counts_pq(heldout),
            cross_fit_A={'fit_photos':[photos[i]['stem'] for i in a],'thresholds':fit_a},
            cross_fit_B={'fit_photos':[photos[i]['stem'] for i in b],'thresholds':fit_b})
        pd.DataFrame([{'photo':p['stem'],'tp':v[0],'fp':v[1],'fn':v[2],'iou_sum':v[3]} for p,v in zip(photos,heldout)]).to_csv(out/'cross_fitted_counts.csv',index=False)
    atomic_json(out/'thresholds.json',report)
    pd.DataFrame(history).to_csv(out/'search.csv',index=False)
    evaluate_cache(replace(c,**best),cat,photos,cache,pre,out/'deployment_threshold_validation')
    return best,report
''', encoding='utf-8')


(PACKAGE / 'config.py').write_text(r'''from dataclasses import asdict, dataclass, field
from pathlib import Path
import hashlib
import json
import os
import random
import numpy as np
import torch


@dataclass
class Config:
    RUN_MODE: str = 'train_calibrate_submit'
    INPUT_ROOT: str = '/kaggle/input'
    OUTPUT_ROOT: str = '/kaggle/working/solar_v1'
    ANNOTATION_PATH: str = ''
    TRAIN_IMAGE_DIR: str = ''
    TEST_IMAGE_DIR: str = ''
    SEED: int = 2026
    N_FOLDS: int = 5
    FOLD_INDEX: int = 0
    INPUT_MODE: str = 'raw_corrected_radial'
    ANNOTATION_MODE: str = 'all_annotators'
    PREPROCESS: bool = True
    DISK_MARGIN: float = 1.01
    DETECTOR_IMAGE_SIZE: int = 1024
    ANCHOR_SIZES: tuple = (16, 32, 64, 128, 256)
    ANCHOR_RATIOS: tuple = (.5, 1., 2.)
    DETECTOR_EPOCHS: int = 10
    TRAIN_DETECTOR: bool = True
    DETECTOR_LR: float = .005
    DETECTOR_ACCUM_STEPS: int = 1
    ENABLE_REFINER: bool = True
    REFINER_SIZE: int = 384
    REFINER_EPOCHS: int = 8
    TRAIN_REFINER: bool = True
    REFINER_BATCH_SIZE: int = 4
    REFINER_ACCUM_STEPS: int = 2
    REFINER_USE_COARSE: bool = True
    REFINER_USE_RADIAL: bool = True
    REFINER_CONTEXT: float = 1.8
    REFINER_MIN_CROP: int = 96
    REFINER_CROPS_PER_PHOTO: int = 8
    REFINER_BOX_JITTER: float = .12
    EMPTY_REFINER_FALLBACK: bool = True
    SEGMENTER_LR: float = .0003
    TVERSKY_ALPHA: float = .3
    TVERSKY_BETA: float = .7
    ENABLE_TILE_BRANCH: bool = False
    TILE_SIZE: int = 512
    TILE_OVERLAP: int = 128
    TILE_EPOCHS: int = 8
    TRAIN_TILE: bool = True
    TILE_BATCH_SIZE: int = 2
    TILE_ACCUM_STEPS: int = 2
    TILE_SAMPLES_PER_PHOTO: int = 4
    TILE_FUSION_WEIGHT: float = .35
    TILE_THRESHOLD: float = .65
    TILE_GROW: int = 8
    TILE_KEEP_UNCLAIMED: bool = False
    TILE_UNCLAIMED_MIN_AREA: int = 64
    DETECTION_THRESHOLD: float = .5
    MASK_THRESHOLD: float = .5
    REFINER_THRESHOLD: float = .65
    MASK_NMS_IOU: float = .5
    MIN_INSTANCE_AREA: int = 32
    POSTPROCESS_MODE: str = 'mask_nms'
    SOFT_NMS_SIGMA: float = .5
    MORPH_CLOSE_RADIUS: int = 0
    FRAGMENT_MODE: str = 'keep'
    FRAGMENT_MIN_AREA: int = 8
    CACHE_SCORE_FLOOR: float = .2
    MAX_DETECTIONS: int = 80
    INFERENCE_BATCH_SIZE: int = 8
    NUM_WORKERS: int = 2
    USE_DDP: bool = True
    AMP: bool = True
    VALIDATE_EVERY: int = 2
    RESUME: bool = True
    PRETRAINED: bool = True
    ALLOW_WEIGHT_DOWNLOAD: bool = True
    PRETRAINED_DIR: str = ''
    CHECKPOINT_DIR: str = ''
    RESUME_SOURCE: str = ''
    REFIT_ALL_DATA: bool = False
    CROSS_FIT_CALIBRATION: bool = True
    CALIBRATION_ROUNDS: int = 2
    CALIBRATION_GRID: dict = field(default_factory=lambda: {
        'DETECTION_THRESHOLD': [.3, .5, .7], 'MASK_THRESHOLD': [.4, .5, .6],
        'REFINER_THRESHOLD': [.5, .65, .8], 'MASK_NMS_IOU': [.3, .5, .7],
        'MIN_INSTANCE_AREA': [16, 32, 64]})
    ENSEMBLE_CACHE_DIRS: tuple = ()
    ENSEMBLE_WEIGHTS: tuple = ()
    ENSEMBLE_MATCH_IOU: float = .55
    QUALITY_WEIGHT: float = 0.
    MAX_CACHE_GB: float = 12.

    def validate(self):
        assert self.RUN_MODE in {'train_calibrate_submit','train','prepare','validate','calibrate','predict'}
        assert self.INPUT_MODE in {'raw_repeat','corrected_repeat','raw_corrected_radial'}
        assert self.ANNOTATION_MODE in {'all_annotators','random_annotator','max_count'}
        assert self.N_FOLDS >= 2 and 0 <= self.FOLD_INDEX < self.N_FOLDS
        assert self.DETECTOR_IMAGE_SIZE in (1024,1280)
        assert self.REFINER_SIZE in (256,384,512)
        assert self.TILE_SIZE % 32 == 0 and 0 <= self.TILE_OVERLAP < self.TILE_SIZE
        assert len(self.ANCHOR_SIZES)==5 and min(self.ANCHOR_SIZES)>0 and min(self.ANCHOR_RATIOS)>0
        assert self.POSTPROCESS_MODE in {'mask_nms','soft_nms','legacy_union'}
        assert self.FRAGMENT_MODE in {'keep','remove_small','split'}
        assert 0 <= self.QUALITY_WEIGHT <= 1 and self.MAX_CACHE_GB>0
        assert self.CACHE_SCORE_FLOOR <= min(self.CALIBRATION_GRID['DETECTION_THRESHOLD'])
        assert 0<=self.CACHE_SCORE_FLOOR<=self.DETECTION_THRESHOLD<=1
        assert all(0<getattr(self,k)<=1 for k in ['MASK_THRESHOLD','REFINER_THRESHOLD','MASK_NMS_IOU','TILE_THRESHOLD'])
        assert self.MIN_INSTANCE_AREA>=1 and self.FRAGMENT_MIN_AREA>=1 and self.SOFT_NMS_SIGMA>0
        assert 0<=self.TILE_FUSION_WEIGHT<=1 and self.TILE_GROW>=0 and self.MORPH_CLOSE_RADIUS>=0
        assert all(getattr(self,k)>=1 for k in ['DETECTOR_EPOCHS','REFINER_EPOCHS','TILE_EPOCHS',
            'REFINER_BATCH_SIZE','TILE_BATCH_SIZE','INFERENCE_BATCH_SIZE','VALIDATE_EVERY',
            'DETECTOR_ACCUM_STEPS','REFINER_ACCUM_STEPS','TILE_ACCUM_STEPS'])
        return self

    def save(self,path):
        atomic_json(path,asdict(self))

    @classmethod
    def load(cls,path):
        return cls(**json.loads(Path(path).read_text())).validate()


def atomic_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(value,indent=2,ensure_ascii=False),encoding='utf-8')
    os.replace(tmp,path)


def digest(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,default=str).encode()).hexdigest()


def file_hash(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()


def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)


def architecture(c,stage):
    common={'version':'1.0.0','stage':stage}
    if stage=='detector':
        common.update(size=c.DETECTOR_IMAGE_SIZE,anchors=c.ANCHOR_SIZES,ratios=c.ANCHOR_RATIOS,
                      input_mode=c.INPUT_MODE,preprocess=c.PREPROCESS,roi_pool=28)
    elif stage=='refiner':
        common.update(size=c.REFINER_SIZE,coarse=c.REFINER_USE_COARSE,radial=c.REFINER_USE_RADIAL)
    else:common.update(size=c.TILE_SIZE,input_mode=c.INPUT_MODE,preprocess=c.PREPROCESS)
    return digest(common)


def autocast(c,device):
    return torch.amp.autocast(device.type,enabled=c.AMP and device.type=='cuda',dtype=torch.float16)
''', encoding='utf-8')


(PACKAGE / 'data.py').write_text(r'''from pathlib import Path
import json
import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from pycocotools.coco import COCO
from pycocotools import mask as cm
from sklearn.model_selection import StratifiedGroupKFold
from .config import atomic_json,digest,file_hash
from .preprocess import SolarPreprocessor,features,radial_window,fit_normalization


def image_files(root):
    return sorted(p for p in Path(root).rglob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.tif','.tiff'})


class Catalog:
    def __init__(self,c):
        self.c=c;root=Path(c.INPUT_ROOT)
        matches=[Path(c.ANNOTATION_PATH)] if c.ANNOTATION_PATH else list(root.rglob('MAGFiLO_1.0_Annotations_kaggle2026_train.json'))
        if len(matches)!=1:raise ValueError('Set ANNOTATION_PATH: expected exactly one training JSON.')
        self.annotation=matches[0]
        self.train_dir=Path(c.TRAIN_IMAGE_DIR) if c.TRAIN_IMAGE_DIR else self.annotation.parent/'train_images'
        self.test_dir=Path(c.TEST_IMAGE_DIR) if c.TEST_IMAGE_DIR else self.annotation.parent.parent/'test/test_images'
        self.coco=COCO(str(self.annotation))
        self.actual_ids={str(k):k for k in self.coco.imgs}
        self.photos={};self.records={}
        for iid,im in self.coco.imgs.items():
            rid=str(iid);stem=Path(im['file_name']).stem
            path=self.train_dir/im['file_name']
            if not path.is_file():path=self.train_dir/Path(im['file_name']).name
            if not path.is_file():raise FileNotFoundError(path)
            if len(stem)<16 or not stem[:14].isdigit():raise ValueError(f'Cannot parse observation date/site: {stem}')
            anns=self.coco.imgToAnns[iid]
            self.records[rid]={'id':rid,'photo':stem,'count':len(anns),'area':sum(float(a.get('area',0)) for a in anns)}
            if stem not in self.photos:self.photos[stem]={'stem':stem,'date':stem[:8],'site':stem[-2:],'path':str(path),'records':[]}
            elif self.photos[stem]['path']!=str(path):raise ValueError('Duplicate observation stem with different paths')
            self.photos[stem]['records'].append(rid)
        self.test_paths=image_files(self.test_dir)
        if len({p.stem for p in self.test_paths})!=len(self.test_paths):raise ValueError('Duplicate test photo stems')
        if set(self.photos)&{p.stem for p in self.test_paths}:raise ValueError('Train/test photo ids overlap')
        self.hash=file_hash(self.annotation)

    def rles(self,rid):
        anns=self.coco.imgToAnns[self.actual_ids[str(rid)]]
        return [self.coco.annToRLE(a) for a in anns if float(cm.area(self.coco.annToRLE(a)))>0]

    def masks(self,rid,size=None,bounds=None):
        result=[]
        for rle in self.rles(rid):
            m=cm.decode(rle)
            if bounds is not None:
                x0,y0,x1,y1=bounds;m=m[y0:y1,x0:x1]
            if size is not None:m=cv2.resize(m,(size,size),interpolation=cv2.INTER_NEAREST)
            if m.any():result.append(m)
        if size is not None:shape=(size,size)
        elif bounds is not None:shape=(bounds[3]-bounds[1],bounds[2]-bounds[0])
        else:
            im=self.coco.imgs[self.actual_ids[str(rid)]];shape=(im['height'],im['width'])
        return np.stack(result).astype(np.uint8) if result else np.zeros((0,*shape),np.uint8)

    def views(self,stem):return {rid:self.rles(rid) for rid in self.photos[stem]['records']}


def prepare(c):
    out=Path(c.OUTPUT_ROOT);out.mkdir(parents=True,exist_ok=True)
    cat=Catalog(c);pre=SolarPreprocessor(out/'preprocessed',c.PREPROCESS)
    photos=[cat.photos[s].copy() for s in sorted(cat.photos)]
    # Union dates connected by duplicate image pixels, so renamed copies also stay together.
    parent={p['date']:p['date'] for p in photos};seen={}
    def find(x):
        while parent[x]!=x:parent[x]=parent[parent[x]];x=parent[x]
        return x
    for i,p in enumerate(photos,1):
        data=pre.load(p['path']);ph=str(data['pixel_hash'])
        if ph in seen:parent[find(p['date'])]=find(seen[ph])
        else:seen[ph]=p['date']
        p['pixel_hash']=ph
        p['disk_cx'],p['disk_cy'],p['disk_radius']=map(float,data['disk'])
        if i%50==0:print(f'Preprocess/cache: {i}/{len(photos)}',flush=True)
    groups=[find(p['date']) for p in photos]
    if len(set(groups))<c.N_FOLDS:raise ValueError('Not enough independent date groups for N_FOLDS')
    folds=np.full(len(photos),-1)
    splitter=StratifiedGroupKFold(c.N_FOLDS,shuffle=True,random_state=c.SEED)
    for fold,(_,val) in enumerate(splitter.split(np.zeros(len(photos)),[p['site'] for p in photos],groups)):
        folds[val]=fold
    for p,f,g in zip(photos,folds,groups):p.update(fold=int(f),group=g,split='val' if f==c.FOLD_INDEX else 'train')
    provenance={'annotation_hash':cat.hash,'seed':c.SEED,'n_folds':c.N_FOLDS,'fold':c.FOLD_INDEX,
                'preprocess_version':pre.VERSION,'preprocess_enabled':c.PREPROCESS,
                'photos':photos}
    provenance['split_hash']=digest({**provenance,'photos':[{k:v for k,v in p.items() if k!='path'} for p in photos]})
    path=out/'manifest.json'
    if path.exists() and json.loads(path.read_text())['split_hash']!=provenance['split_hash']:
        raise ValueError('Existing manifest differs. Choose a new OUTPUT_ROOT for a different dataset/split.')
    atomic_json(path,provenance)
    rows=[{**{k:v for k,v in p.items() if k!='records'},'record_id':rid,
           'n_views':len(p['records']),'n_instances':cat.records[rid]['count']} for p in photos for rid in p['records']]
    pd.DataFrame(rows).to_csv(out/'all_annotators_manifest.csv',index=False)
    pd.crosstab(pd.Series([p['site'] for p in photos],name='site'),pd.Series([p['split'] for p in photos],name='split')).to_csv(out/'site_split.csv')
    train=[p for p in photos if p['split']=='train']
    norm=fit_normalization(c,pre,train);norm['split_hash']=provenance['split_hash']
    atomic_json(out/'normalization.json',norm)
    atomic_json(out/'data_profile.json',{'n_photos':len(photos),'n_annotation_views':len(cat.records),
        'n_annotations':len(cat.coco.anns),'train_photos':len(train),'val_photos':len(photos)-len(train),
        'n_test_photos':len(cat.test_paths),'station_balance':'StratifiedGroupKFold by site, groups join dates and duplicate pixels',
        'train_val_dates_disjoint':not({p['date'] for p in train}&{p['date'] for p in photos if p['split']=='val'})})
    return cat,provenance,norm


def load_manifest(c):return json.loads((Path(c.OUTPUT_ROOT)/'manifest.json').read_text())


def selected_record(cat,p,mode,rng,epoch=0,slot=0):
    ids=p['records']
    if mode=='max_count':return max(ids,key=lambda r:(cat.records[r]['count'],cat.records[r]['area'],r))
    if mode=='all_annotators':return ids[(epoch+slot)%len(ids)]
    if mode=='random_annotator':
        # Same record for this photo throughout an epoch, independent of workers/ranks.
        seed=int(digest([cat.c.SEED,p['stem'],epoch])[:8],16)
        return ids[np.random.default_rng(seed).integers(len(ids))]
    raise ValueError(mode)


def spatial_augment(x,m,rng):
    k=int(rng.integers(4));x=np.rot90(x,k,axes=(-2,-1));m=np.rot90(m,k,axes=(-2,-1))
    if rng.random()<.5:x=np.flip(x,-1);m=np.flip(m,-1)
    return np.ascontiguousarray(x),np.ascontiguousarray(m)


def photometric_features(x,mode,rng):
    """Reference gamma/contrast/gain/blur/noise, applied only to intensity channels."""
    count=2 if mode=='raw_corrected_radial' else 1
    gamma=rng.uniform(.8,1.25) if rng.random()<.7 else 1.
    contrast,gain=(rng.uniform(.85,1.15),rng.uniform(.9,1.1)) if rng.random()<.7 else (1.,1.)
    blur=rng.uniform(.3,.8) if rng.random()<.15 else 0.
    noise=rng.uniform(.003,.015) if rng.random()<.25 else 0.
    for i in range(count):
        image=x[i].copy();valid=image>.01
        if not valid.any():continue
        image[valid]**=gamma;mean=image[valid].mean()
        image[valid]=((image[valid]-mean)*contrast+mean)*gain
        if blur:image[valid]=cv2.GaussianBlur(image,(0,0),blur)[valid]
        if noise:image[valid]+=rng.normal(0,noise,int(valid.sum()))
        x[i]=np.clip(image,0,1)
    if count==1:x[1]=x[0];x[2]=x[0]
    return x


def boxes_from_masks(masks):
    result=[]
    for m in masks:
        y,x=np.where(m);result.append([x.min(),y.min(),x.max()+1,y.max()+1])
    return np.asarray(result,np.float32).reshape(-1,4)


class EpochDataset(Dataset):
    @property
    def epoch(self):return int(self._epoch.item())

    @epoch.setter
    def epoch(self,value):
        if hasattr(self,'_epoch'):self._epoch.fill_(value)
        else:self._epoch=torch.tensor(value,dtype=torch.long).share_memory_()


class DetectorDataset(EpochDataset):
    def __init__(self,c,cat,photos):
        self.c,self.cat,self.photos=c,cat,photos;self.epoch=0
        self.pre=SolarPreprocessor(Path(c.OUTPUT_ROOT)/'preprocessed',c.PREPROCESS)
        if c.ANNOTATION_MODE=='all_annotators':self.items=[(p,r) for p in photos for r in p['records']]
        else:self.items=[(p,None) for p in photos]

    def __len__(self):return len(self.items)

    def __getitem__(self,index):
        p,r=self.items[index];rng=np.random.default_rng(self.c.SEED+self.epoch*1000003+index)
        if r is None:r=selected_record(self.cat,p,self.c.ANNOTATION_MODE,rng,self.epoch)
        x=features(self.pre.load(p['path']),self.c.INPUT_MODE,size=self.c.DETECTOR_IMAGE_SIZE)
        m=self.cat.masks(r,size=self.c.DETECTOR_IMAGE_SIZE)
        x,m=spatial_augment(x,m,rng)
        x=photometric_features(x,self.c.INPUT_MODE,rng)
        weight=len(self.items)/(len(self.photos)*len(p['records'])) if self.c.ANNOTATION_MODE=='all_annotators' else 1.
        target={'boxes':torch.from_numpy(boxes_from_masks(m)),'labels':torch.ones(len(m),dtype=torch.long),
                'masks':torch.from_numpy(m),'image_id':torch.tensor([index]),
                'area':torch.from_numpy(m.sum((1,2)).astype(np.float32)),'iscrowd':torch.zeros(len(m),dtype=torch.long)}
        return torch.from_numpy(x),target,float(weight)


def square_box(box,shape,context=1.8,minimum=96,rng=None,jitter=0.):
    x0,y0,x1,y1=map(float,box);h,w=shape
    side=max(x1-x0,y1-y0)*context;cx=(x0+x1)/2;cy=(y0+y1)/2
    if rng is not None and jitter:
        cx+=rng.uniform(-jitter,jitter)*side;cy+=rng.uniform(-jitter,jitter)*side
        side*=rng.uniform(.85,1.2)
    side=int(min(max(np.ceil(side),minimum),h,w))
    x=int(np.clip(round(cx-side/2),0,w-side));y=int(np.clip(round(cy-side/2),0,h-side))
    return x,y,x+side,y+side


def roi_channels(data,bounds,size,coarse,c):
    x=features(data,'raw_repeat',bounds,size)
    if c.REFINER_USE_COARSE:x=np.concatenate([x,cv2.resize(coarse.astype(np.float32),(size,size),interpolation=cv2.INTER_LINEAR)[None]],0)
    if c.REFINER_USE_RADIAL:x=np.concatenate([x,cv2.resize(radial_window(data,bounds),(size,size),interpolation=cv2.INTER_LINEAR)[None]],0)
    return np.ascontiguousarray(x)


class SegDataset(EpochDataset):
    def __init__(self,c,cat,photos,stage):
        self.c,self.cat,self.photos,self.stage=c,cat,photos,stage;self.epoch=0
        self.pre=SolarPreprocessor(Path(c.OUTPUT_ROOT)/'preprocessed',c.PREPROCESS)
        self.samples=c.REFINER_CROPS_PER_PHOTO if stage=='refiner' else c.TILE_SAMPLES_PER_PHOTO
        # Retain negative photos for tile training; refiner needs one instance.
        if stage=='refiner':self.photos=[p for p in photos if any(cat.records[r]['count'] for r in p['records'])]
        if not self.photos:raise ValueError(f'No training examples for {stage}')

    def __len__(self):return len(self.photos)*self.samples

    def __getitem__(self,index):
        p=self.photos[index//self.samples];slot=index%self.samples
        rng=np.random.default_rng(self.c.SEED+self.epoch*1000003+index)
        rid=selected_record(self.cat,p,self.c.ANNOTATION_MODE,rng,self.epoch,slot)
        data=self.pre.load(p['path']);h,w=data['raw'].shape
        if self.stage=='refiner':
            if not self.cat.records[rid]['count']:
                available=[r for r in p['records'] if self.cat.records[r]['count']]
                rid=available[int(rng.integers(len(available)))]
            rles=self.cat.rles(rid);r=rles[int(rng.integers(len(rles)))];m=cm.decode(r)
            bx,by,bw,bh=cm.toBbox(r)
            bounds=square_box([bx,by,bx+bw,by+bh],(h,w),self.c.REFINER_CONTEXT,self.c.REFINER_MIN_CROP,rng,self.c.REFINER_BOX_JITTER)
            x0,y0,x1,y1=bounds;target=m[y0:y1,x0:x1]
            small=cv2.resize(target.astype(np.float32),(max(8,(x1-x0)//8),max(8,(y1-y0)//8)),interpolation=cv2.INTER_AREA)
            coarse=cv2.resize(small.astype(np.float32),(x1-x0,y1-y0),interpolation=cv2.INTER_LINEAR)
            if rng.random()<.3:coarse=cv2.erode(coarse,np.ones((3,3),np.uint8))
            if rng.random()<.3:coarse=cv2.dilate(coarse,np.ones((3,3),np.uint8))
            if rng.random()<.1:coarse*=0
            size=self.c.REFINER_SIZE;x=roi_channels(data,bounds,size,coarse,self.c)
            y=cv2.resize(target,(size,size),interpolation=cv2.INTER_NEAREST)[None].astype(np.float32)
        else:
            size=self.c.TILE_SIZE
            if h<size or w<size:raise ValueError('TILE_SIZE exceeds native image size')
            rles=self.cat.rles(rid)
            if rles and rng.random()<.7:
                bx,by,bw,bh=cm.toBbox(rles[int(rng.integers(len(rles)))])
                cx=rng.uniform(bx,bx+bw);cy=rng.uniform(by,by+bh)
                x0=int(np.clip(cx-size/2,0,w-size));y0=int(np.clip(cy-size/2,0,h-size))
            else:x0=int(rng.integers(w-size+1));y0=int(rng.integers(h-size+1))
            bounds=(x0,y0,x0+size,y0+size)
            x=features(data,self.c.INPUT_MODE,bounds,size)
            masks=self.cat.masks(rid,bounds=bounds)
            y=(masks.any(0) if len(masks) else np.zeros((size,size),bool))[None].astype(np.float32)
        x,y=spatial_augment(x,y,rng)
        if self.stage=='refiner':
            if rng.random()<.5:x[:3]=x[:3]**rng.uniform(.85,1.2)
            if rng.random()<.5:x[:3]=np.clip(x[:3]*rng.uniform(.9,1.1),0,1)
        else:x=photometric_features(x,self.c.INPUT_MODE,rng)
        return torch.from_numpy(x),torch.from_numpy(y)


def detection_collate(batch):return tuple(zip(*batch))
''', encoding='utf-8')


(PACKAGE / 'disk.py').write_text(r'''"""Solar disk geometry and limb-darkening correction.

Two preprocessing steps that matter more than the choice of model:

1. **Disk masking.** Everything outside the solar limb is sky and detector
   noise. It is uniformly dark, so any "filaments are dark" rule fires on it.
   Masking it away removes a large class of false positives for free -- and
   under Panoptic Quality every false positive costs 0.5 in the denominator.

2. **Limb-darkening correction.** A full-disk H-alpha image is systematically
   darker toward the limb, so a single global darkness threshold is far too
   eager near the edge and far too shy at disk centre. Dividing by the median
   radial intensity profile makes one threshold mean the same thing everywhere.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np


@dataclass(frozen=True)
class Disk:
    cx: float
    cy: float
    radius: float

    def mask(self, shape: tuple[int, int], shrink: float = 0.98) -> np.ndarray:
        """Boolean mask of the disk, shrunk slightly to drop the noisy limb."""
        height, width = shape
        ys, xs = np.ogrid[:height, :width]
        r2 = (xs - self.cx) ** 2 + (ys - self.cy) ** 2
        return r2 <= (self.radius * shrink) ** 2

    def radius_map(self, shape: tuple[int, int]) -> np.ndarray:
        """Normalised radial distance (1.0 at the limb)."""
        return self.radius_window(0, 0, *shape)

    def radius_window(self, y0: int, x0: int, height: int, width: int) -> np.ndarray:
        """``radius_map`` for rows ``y0:y0+height`` and columns ``x0:x0+width`` only."""
        ys, xs = np.ogrid[y0 : y0 + height, x0 : x0 + width]
        return np.sqrt((xs - self.cx) ** 2 + (ys - self.cy) ** 2) / max(self.radius, 1.0)


def read_image(path: str | Path) -> np.ndarray:
    """Load a GONG H-alpha JPEG as a single-channel uint8 array.

    The competition data is grayscale; reading it as BGR and averaging would
    only add rounding noise.
    """
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise FileNotFoundError(f"could not read image: {path}")
    return image


#: GONG H-alpha frames are usually registered to a standard geometry: the
#: disk centred and about 0.439 frame-widths in radius (899 px at 2048). Many
#: frames are not, so this is only one of several starting guesses.
_STANDARD_RADIUS_FRACTION = 0.439
#: Rays cast from a candidate centre when locating the limb.
_N_RAYS = 720
#: How far inside each starting guess to look as well, in px at 2048. The halo
#: can reach 150 px beyond the limb, so a guess locked onto it needs this.
_INWARD_OFFSETS = (0.0, 50.0, 100.0, 150.0)
#: A ray "agrees" with a circle if its sharpest edge lies within this many px
#: (at 2048) of it. Fixed, not data-adaptive, so scattered edges score low.
_AGREEMENT_TOLERANCE = 3.0
#: A circle counts as well supported if this fraction of the best circle's
#: agreeing rays also agree with it.
_SUPPORT_RATIO = 0.8


def detect_disk(image: np.ndarray) -> Disk:
    """Locate the solar disk by its limb.

    GONG frames carry a smooth "halo" annulus outside the limb -- scattered
    light, and the area the frame covered before it was registered -- that can
    be nearly as bright as the disk itself, and its outer border is a crisp
    circle too. Thresholding the image, which this function used to rely on
    alone, locks onto that border: about 35 px beyond the limb on a typical
    frame and up to 150 px on the worst.

    Instead, several starting circles (an intensity threshold, a texture
    threshold -- the disk is textured, the halo smooth -- and the standard GONG
    geometry), each also shrunk by up to 150 px, are snapped to the sharpest
    radial edge nearby and scored by how many rays agree with the result. The
    halo always lies outside the limb, so of the well-supported circles the
    smallest is the limb.
    """
    scale = min(image.shape) / 2048.0
    blurred = cv2.GaussianBlur(image.astype(np.float32), (0, 0), max(1.5 * scale, 0.8))
    seeds = [_threshold_disk(image), _texture_disk(image), _standard_disk(image)]

    fits: list[tuple[Disk, float]] = []
    for seed in seeds:
        if seed is None:
            continue
        for offset in _INWARD_OFFSETS:
            start = Disk(seed.cx, seed.cy, seed.radius - offset * scale)
            if start.radius <= 0:
                continue
            disk, _ = _refine_limb(blurred, start, inside=60.0, outside=20.0)
            # A second, narrow pass lets the centre settle once the radius is close.
            fits.append(_refine_limb(blurred, disk, inside=10.0, outside=10.0))

    best_support = max((support for _, support in fits), default=0.0)
    # Fewer than half the rays agreeing means nothing found a clean limb; the
    # plain threshold circle is then the least surprising answer.
    if best_support < 0.5:
        return seeds[0]
    supported = [disk for disk, support in fits if support >= _SUPPORT_RATIO * best_support]
    return min(supported, key=lambda disk: disk.radius)


def _fit_circle(xs: np.ndarray, ys: np.ndarray, iterations: int = 4) -> tuple[Disk, np.ndarray]:
    """Least-squares (Kasa) circle fit, refitted without outliers.

    Returns the circle and a boolean inlier mask over the input points.
    """
    keep = np.ones(xs.size, dtype=bool)
    disk = Disk(0.0, 0.0, 1.0)
    for _ in range(iterations):
        if keep.sum() < 3:
            break
        design = np.column_stack([xs[keep], ys[keep], np.ones(int(keep.sum()))])
        target = -(xs[keep] ** 2 + ys[keep] ** 2)
        d, e, f = np.linalg.lstsq(design, target, rcond=None)[0]
        cx, cy = -d / 2.0, -e / 2.0
        disk = Disk(float(cx), float(cy), float(np.sqrt(max(cx * cx + cy * cy - f, 1.0))))
        residual = np.hypot(xs - disk.cx, ys - disk.cy) - disk.radius
        spread = 1.4826 * np.median(np.abs(residual[keep] - np.median(residual[keep])))
        keep = np.abs(residual) <= max(3.0 * spread, 2.0)
    return disk, keep


def _refine_limb(
    blurred: np.ndarray, start: Disk, inside: float, outside: float
) -> tuple[Disk, float]:
    """Snap ``start`` to the sharpest radial edge within the search band.

    ``blurred`` is the lightly smoothed float image. Band lengths are in px at
    2048 and scale with the frame, so the same code works on small test
    images. Returns the refined circle and the fraction of all rays whose
    sharpest edge lies within ``_AGREEMENT_TOLERANCE`` of it.
    """
    height, width = blurred.shape
    scale = min(height, width) / 2048.0
    theta = np.linspace(0.0, 2.0 * np.pi, _N_RAYS, endpoint=False)[:, None]
    radii = np.arange(start.radius - inside * scale, start.radius + outside * scale + 1.0, 1.0)
    map_x = (start.cx + radii[None, :] * np.cos(theta)).astype(np.float32)
    map_y = (start.cy + radii[None, :] * np.sin(theta)).astype(np.float32)
    profiles = cv2.remap(blurred, map_x, map_y, cv2.INTER_LINEAR,
                         borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    in_frame = (map_x >= 1) & (map_x <= width - 2) & (map_y >= 1) & (map_y <= height - 2)
    gradient = np.abs(np.gradient(profiles, axis=1))
    gradient[~in_frame] = -1.0
    edge = np.argmax(gradient, axis=1)
    valid = gradient[np.arange(_N_RAYS), edge] > 0
    # A real edge is a peak inside the band. A maximum on the band's boundary
    # is just the smooth limb-darkening slope rising toward it -- perfectly
    # circular, so without this it would pass for a well-supported limb.
    valid &= (edge > 1) & (edge < radii.size - 2)
    if valid.sum() < 3:
        return start, 0.0
    # Sub-pixel peak position from a parabola through the peak and its two
    # neighbours: the flattening downstream is sensitive to fractions of a
    # pixel in the radius.
    rows = np.flatnonzero(valid)
    peak = edge[valid]
    before, at, after = (gradient[rows, peak - 1], gradient[rows, peak],
                         gradient[rows, peak + 1])
    curvature = before - 2.0 * at + after
    shift = np.divide(0.5 * (before - after), curvature,
                      out=np.zeros_like(at), where=curvature < 0)
    r = radii[peak] + np.clip(shift, -0.5, 0.5)
    xs = start.cx + r * np.cos(theta[valid, 0])
    ys = start.cy + r * np.sin(theta[valid, 0])
    disk, _ = _fit_circle(xs, ys)
    residual = np.hypot(xs - disk.cx, ys - disk.cy) - disk.radius
    agreeing = np.abs(residual) <= max(_AGREEMENT_TOLERANCE * scale, 1.0)
    return disk, float(agreeing.sum()) / _N_RAYS


def _standard_disk(image: np.ndarray) -> Disk:
    height, width = image.shape
    return Disk((width - 1) / 2.0, height / 2.0, min(height, width) * _STANDARD_RADIUS_FRACTION)


def _texture_disk(image: np.ndarray) -> Disk | None:
    """Disk from local texture: chromospheric structure is busy, the halo smooth."""
    scale = min(image.shape) / 2048.0
    img = image.astype(np.float32)
    detail = np.abs(img - cv2.GaussianBlur(img, (0, 0), max(3.0 * scale, 1.0)))
    texture = cv2.GaussianBlur(detail, (0, 0), max(15.0 * scale, 2.0))
    texture_u8 = np.clip(texture * (255.0 / max(float(texture.max()), 1e-6)), 0, 255)
    _, binary = cv2.threshold(texture_u8.astype(np.uint8), 0, 1,
                              cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    if n_labels <= 1:
        return None
    largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    component = (labels == largest).astype(np.uint8)
    # Fill the component's interior: smooth patches on the disk (and the whole
    # interior of a texture-free test image) leave holes.
    flood = component.copy()
    cv2.floodFill(flood, np.zeros((flood.shape[0] + 2, flood.shape[1] + 2), np.uint8), (0, 0), 1)
    component = component | (1 - flood)
    contours, _ = cv2.findContours(component, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    points = max(contours, key=cv2.contourArea)[:, 0, :].astype(np.float64)
    if len(points) < 3:
        return None
    disk, _ = _fit_circle(points[:, 0], points[:, 1])
    return disk


def _threshold_disk(image: np.ndarray) -> Disk:
    """Disk from an intensity threshold: the brightest large component.

    Reliable for the centre of a clean frame, but the halo annulus usually
    passes the threshold too, so the radius it gives is an upper bound.
    """
    blurred = cv2.GaussianBlur(image, (0, 0), sigmaX=5)
    threshold, binary = cv2.threshold(
        blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    binary = cv2.morphologyEx(
        binary, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (31, 31))
    )

    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    if n_labels <= 1:
        # Degenerate image: fall back to the nominal full-frame disk.
        height, width = image.shape
        return Disk(width / 2, height / 2, min(height, width) * 0.45)

    largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    component = (labels == largest).astype(np.uint8)

    contours, _ = cv2.findContours(component, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contour = max(contours, key=cv2.contourArea)
    (cx, cy), radius = cv2.minEnclosingCircle(contour)
    return Disk(float(cx), float(cy), float(radius))


def limb_darkening_profile(
    image: np.ndarray, disk: Disk, n_bins: int = 128, smooth: int = 9
) -> np.ndarray:
    """Median intensity as a function of normalised radius."""
    radius = disk.radius_map(image.shape)
    inside = radius <= 1.0
    bins = np.clip((radius[inside] * n_bins).astype(np.int32), 0, n_bins - 1)
    values = image[inside].astype(np.float32)

    profile = np.ones(n_bins, dtype=np.float32)
    order = np.argsort(bins, kind="stable")
    bins_sorted, values_sorted = bins[order], values[order]
    edges = np.searchsorted(bins_sorted, np.arange(n_bins + 1))
    for b in range(n_bins):
        lo, hi = edges[b], edges[b + 1]
        if hi > lo:
            profile[b] = np.median(values_sorted[lo:hi])

    # Fill empty bins, then smooth so the correction has no step artefacts.
    valid = profile > 0
    if valid.any():
        profile = np.interp(np.arange(n_bins), np.flatnonzero(valid), profile[valid])
    if smooth > 1:
        kernel = np.ones(smooth, dtype=np.float32) / smooth
        profile = np.convolve(profile, kernel, mode="same")
    profile[profile <= 0] = 1.0
    return profile


def flatten(image: np.ndarray, disk: Disk, n_bins: int = 128) -> np.ndarray:
    """Divide out limb darkening.

    Returns a float32 image where 1.0 is "typical quiet Sun at this radius" and
    filaments sit clearly below 1.0, independent of distance from disk centre.
    """
    profile = limb_darkening_profile(image, disk, n_bins=n_bins)
    radius = disk.radius_map(image.shape)
    bin_index = np.clip((radius * n_bins).astype(np.int32), 0, n_bins - 1)
    expected = profile[bin_index]
    flat = image.astype(np.float32) / np.maximum(expected, 1e-3)
    flat[radius > 1.0] = 1.0
    return flat

''', encoding='utf-8')


(PACKAGE / 'driver.py').write_text(r'''"""The public entry point: prepare -> train -> PQ calibrate -> test -> submission.csv."""
import argparse
from dataclasses import replace
from pathlib import Path
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import time
from types import SimpleNamespace
import cv2
import torch
from .config import Config,atomic_json,seed_all
from .data import prepare,image_files
from .preprocess import SolarPreprocessor
from .models import checkpoint_path,load_model,warm_pretrained
from .inference import PredictionCache,cache_photos
from .evaluation import evaluate_cache
from .calibration import calibrate
from .postprocess import postprocess
from .submission import SubmissionWriter,export_instances


def train_subprocess(c,config_path,stage,all_data=False,epochs=None):
    count=min(2,torch.cuda.device_count()) if c.USE_DDP else 1
    if not torch.cuda.is_available():raise RuntimeError('Enable Kaggle GPU T4 x2 before training')
    command=[sys.executable]
    if count>1:command+=['-m','torch.distributed.run','--standalone',f'--nproc_per_node={count}']
    command+=['-m','solar_v1.train','--config',str(config_path),'--stage',stage]
    if all_data:command+=['--all-data']
    if epochs is not None:command+=['--epochs',str(epochs)]
    env=os.environ.copy();package_root=str(Path(__file__).resolve().parent.parent)
    env['PYTHONPATH']=package_root+os.pathsep+env.get('PYTHONPATH','')
    env['OMP_NUM_THREADS']='2'
    print('Training:',stage,'all data' if all_data else 'date-grouped train split',f'GPUs={count}',flush=True)
    subprocess.run(command,check=True,env=env)


def make_cache(c,norm,photos,pre,refiner=True,tile=True,all_data=False,validation=True):
    device=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    if device.type!='cuda':raise RuntimeError('Real inference is intended for Kaggle GPU')
    detpath=checkpoint_path(c,'detector',all_data=all_data)
    refpath=checkpoint_path(c,'refiner',all_data=all_data) if refiner and c.ENABLE_REFINER else None
    tilepath=checkpoint_path(c,'tile',all_data=all_data) if tile and c.ENABLE_TILE_BRANCH else None
    cache=PredictionCache(c,detpath,refpath,tilepath,norm)
    detector,ds=load_model(c,norm,'detector',device,detpath)
    ref,rs=load_model(c,norm,'refiner',device,refpath) if refpath else (None,None)
    tiled,ts=load_model(c,norm,'tile',device,tilepath) if tilepath else (None,None)
    trained=set()
    for state in [ds,rs,ts]:
        if state:trained.update(state['trained_photo_ids'])
    for folder in c.ENSEMBLE_CACHE_DIRS:
        metadata=json.loads((Path(folder)/'ensemble_manifest.json').read_text())
        trained.update(metadata['trained_photo_ids'])
    if validation and (trained&{p['stem'] for p in photos}):raise ValueError('Model training photos overlap requested validation photos')
    times=cache_photos(c,photos,pre,cache,detector,ref,tiled,device)
    atomic_json(cache.root/'timing.json',{'new_forward_photos':len(times),'seconds':times,'candidate_cap':c.MAX_DETECTIONS})
    atomic_json(cache.root/'ensemble_manifest.json',{'trained_photo_ids':sorted(trained),'cache_signature':cache.signature,
        'export_format':'per-photo JSON of independently scored instances; COCO compressed RLE'})
    del detector,ref,tiled;torch.cuda.empty_cache()
    return cache,sorted(trained)


def submit(c,cat,norm,pre):
    paths=cat.test_paths
    if not paths:raise ValueError('No test images found: set TEST_IMAGE_DIR')
    photos=[{'stem':p.stem,'path':str(p),'site':p.stem[-2:],'date':p.stem[:8]} for p in paths]
    cache,trained=make_cache(c,norm,photos,pre,all_data=c.REFIT_ALL_DATA,validation=False)
    export=Path(c.OUTPUT_ROOT)/'test_instances';export.mkdir(exist_ok=True)
    atomic_json(export/'ensemble_manifest.json',{'trained_photo_ids':trained,'cache_signature':cache.signature})
    writer=SubmissionWriter('/kaggle/working/submission.csv',[p.stem for p in paths],coverage_path=Path(c.OUTPUT_ROOT)/'submission_coverage.json')
    for p in photos:
        data=pre.load(p['path']);pack=cache.load(p['stem'],data)
        result=postprocess(pack,c,data,p['stem'],False)
        writer.add(p['stem'],result);export_instances(export,p['stem'],result)
    frame=writer.finish();print(f'Saved /kaggle/working/submission.csv: {len(frame)} instances / {len(paths)} processed photos',flush=True)
    return {'submission':'/kaggle/working/submission.csv','n_test_photos':len(paths),'n_instances':len(frame)}


def run(c):
    c.validate()
    if not Path('/kaggle/input').is_dir():raise RuntimeError('Run this complete pipeline on Kaggle; competition data stays there.')
    if c.RUN_MODE!='prepare' and not torch.cuda.is_available():raise RuntimeError('Enable a Kaggle GPU accelerator')
    start=time.perf_counter();seed_all(c.SEED);torch.set_num_threads(2);cv2.setNumThreads(0)
    out=Path(c.OUTPUT_ROOT);out.mkdir(parents=True,exist_ok=True)
    if c.RESUME_SOURCE:
        source=Path(c.RESUME_SOURCE)
        if source.resolve()==out.resolve():raise ValueError('RESUME_SOURCE must differ from OUTPUT_ROOT')
        if not (source/'manifest.json').is_file():raise FileNotFoundError('RESUME_SOURCE must be a saved V1 output folder')
        for name in ['manifest.json','normalization.json']:
            if not (out/name).exists():shutil.copy2(source/name,out/name)
        for folder in ['checkpoints','calibration']:
            if (source/folder).is_dir():
                for saved in (source/folder).rglob('*'):
                    if saved.is_file():
                        target=out/folder/saved.relative_to(source/folder)
                        if not target.exists():target.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(saved,target)
    if c.CHECKPOINT_DIR and c.RUN_MODE in {'train','train_calibrate_submit'}:
        raise ValueError('CHECKPOINT_DIR is read-only inference input. Use RESUME_SOURCE to restore training into OUTPUT_ROOT.')
    c.save(out/'config.json')
    versions={}
    for package in ['torch','torchvision','numpy','pycocotools','pandas','scikit-learn']:
        versions[package]=importlib.metadata.version(package)
    versions['opencv']=cv2.__version__
    atomic_json(out/'runtime.json',{'python':platform.python_version(),'packages':versions,
        'cuda':torch.version.cuda,'gpus':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]})
    pre=SolarPreprocessor(out/'preprocessed',c.PREPROCESS)
    if c.RUN_MODE=='predict':
        # Inference requires saved training metadata, not training images/labels.
        norm_path=out/'normalization.json';manifest_path=out/'manifest.json'
        if not norm_path.is_file() or not manifest_path.is_file():
            raise FileNotFoundError('Predict requires saved V1 metadata. Set RESUME_SOURCE to the completed V1 output folder.')
        norm=json.loads(norm_path.read_text());manifest=json.loads(manifest_path.read_text())
        if c.TEST_IMAGE_DIR:test_dir=Path(c.TEST_IMAGE_DIR)
        else:
            dirs=[p for p in Path(c.INPUT_ROOT).rglob('test_images') if p.is_dir()]
            if len(dirs)!=1:raise ValueError('Set TEST_IMAGE_DIR: expected exactly one test_images folder')
            test_dir=dirs[0]
        cat=SimpleNamespace(test_paths=image_files(test_dir));val=[]
    else:
        cat,manifest,norm=prepare(c)
        val=[p for p in manifest['photos'] if p['split']=='val']
    if c.RUN_MODE=='prepare':return
    training=c.RUN_MODE in {'train_calibrate_submit','train'}
    if training:
        warm_pretrained(c)
        if c.TRAIN_DETECTOR:train_subprocess(c,out/'config.json','detector')
        if c.ENABLE_REFINER and c.TRAIN_REFINER:
            make_cache(c,norm,val,pre,refiner=False,tile=False)
            train_subprocess(c,out/'config.json','refiner')
        if c.ENABLE_TILE_BRANCH and c.TRAIN_TILE:
            make_cache(c,norm,val,pre,tile=False)
            train_subprocess(c,out/'config.json','tile')
    report={}
    if c.RUN_MODE!='predict':
        cache,trained=make_cache(c,norm,val,pre)
        base=evaluate_cache(c,cat,val,cache,pre,out/'validation_default_thresholds')
        report['validation_default_thresholds']=base.summary()
        # All stages are scored on the same photos/annotator records.
        stage_rows=[]
        for name,ref,tile in [('detector_only',False,False),('detector_refiner',c.ENABLE_REFINER,False),('complete',c.ENABLE_REFINER,c.ENABLE_TILE_BRANCH)]:
            trial=replace(c,ENABLE_REFINER=ref,ENABLE_TILE_BRANCH=tile)
            score=evaluate_cache(trial,cat,val,cache,pre).summary()
            stage_rows.append({'stage':name,'refiner_enabled':ref,'tile_enabled':tile,**score})
        import pandas as pd
        pd.DataFrame(stage_rows).to_csv(out/'stage_metrics.csv',index=False)
        export=out/'validation_instances';export.mkdir(exist_ok=True)
        atomic_json(export/'ensemble_manifest.json',{'trained_photo_ids':trained,'cache_signature':cache.signature})
        if c.RUN_MODE in {'train_calibrate_submit','calibrate'}:
            params,calibration=calibrate(c,cat,val,cache,pre);c=replace(c,**params);report['calibration']=calibration
        for p in val:export_instances(export,p['stem'],postprocess(cache.load(p['stem']),c,pre.load(p['path']),p['stem'],True))
    if c.RUN_MODE in {'train_calibrate_submit','predict'}:
        if c.RUN_MODE=='predict':
            path=out/'calibration/thresholds.json'
            if not path.is_file():raise FileNotFoundError('Predict mode requires calibration/thresholds.json from this V1 run')
            c=replace(c,**json.loads(path.read_text())['deployment_thresholds'])
        if c.REFIT_ALL_DATA and training:
            # Thresholds freeze before all-data training; never score these weights on val.
            for stage,enabled in [('detector',True),('refiner',c.ENABLE_REFINER),('tile',c.ENABLE_TILE_BRANCH)]:
                if enabled:
                    epoch=torch.load(checkpoint_path(c,stage),map_location='cpu',weights_only=True)['epoch']
                    train_subprocess(c,out/'config.json',stage,True,epoch)
        report.update(submit(c,cat,norm,pre))
    report.update(elapsed_minutes=(time.perf_counter()-start)/60,mode=c.RUN_MODE,
                  real_gpu_run=True,split_hash=manifest['split_hash'],all_data_submission=c.REFIT_ALL_DATA and 'submission' in report)
    atomic_json(out/'run_summary.json',report)
    print(json.dumps(report,indent=2),flush=True)


if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('--config',required=True);args=p.parse_args()
    run(Config.load(args.config))
''', encoding='utf-8')


(PACKAGE / 'evaluation.py').write_text(r'''from collections import defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
from pycocotools import mask as cm
from .pq import Totals,from_iou,pairwise_iou,require_disjoint
from .config import atomic_json
from .postprocess import postprocess


class Evaluator:
    def __init__(self):
        self.total=Totals();self.per_photo={};self.per_station=defaultdict(Totals)
        self.matches=[];self.gt_rows=[];self.n_records=0

    def add(self,photo,views,instances,shape):
        pred=[x.rle for x in instances];require_disjoint(pred,shape)
        subtotal=Totals()
        for rid,gt in views.items():
            self.n_records+=1;iou=pairwise_iou(gt,pred);counts=from_iou(iou);subtotal=subtotal+counts
            best=iou.max(1) if iou.shape[1] else np.zeros(len(gt))
            for i,g in enumerate(gt):
                area=float(cm.area(g));bin_name='<250' if area<250 else '250-999' if area<1000 else '1000-3999' if area<4000 else '>=4000'
                self.gt_rows.append({'photo':photo['stem'],'record':rid,'gt_index':i,'area':area,'area_bin':bin_name,
                                     'site':photo['site'],'best_iou':float(best[i]),'recalled':int(best[i]>.5)})
            for i,j in zip(*np.where(iou>.5)):
                self.matches.append({'photo':photo['stem'],'record':rid,'gt_index':int(i),'prediction_index':int(j),'iou':float(iou[i,j])})
        self.total=self.total+subtotal;self.per_photo[photo['stem']]=subtotal.summary()
        self.per_station[photo['site']]=self.per_station[photo['site']]+subtotal
        return subtotal

    def summary(self):
        out=self.total.summary();out['rq']=out['dq'];out['n_photos']=len(self.per_photo);out['n_annotation_views']=self.n_records
        return out

    def save(self,root):
        root=Path(root);root.mkdir(parents=True,exist_ok=True)
        atomic_json(root/'metrics.json',self.summary())
        pd.DataFrame([{'photo':s,**v} for s,v in self.per_photo.items()]).to_csv(root/'per_photo.csv',index=False)
        pd.DataFrame([{'site':s,**t.summary(),'rq':t.summary()['dq']} for s,t in self.per_station.items()]).to_csv(root/'per_station.csv',index=False)
        pd.DataFrame(self.matches,columns=['photo','record','gt_index','prediction_index','iou']).to_csv(root/'matched_ious.csv',index=False)
        frame=pd.DataFrame(self.gt_rows,columns=['photo','record','gt_index','area','area_bin','site','best_iou','recalled'])
        frame.to_csv(root/'gt_errors.csv',index=False)
        if len(frame):
            slices=frame.groupby('area_bin',observed=True).agg(n_gt=('recalled','size'),tp=('recalled','sum'),recall=('recalled','mean')).reset_index()
            slices['fn']=slices.n_gt-slices.tp
        else:slices=pd.DataFrame(columns=['area_bin','n_gt','tp','recall','fn'])
        slices.to_csv(root/'recall_by_area.csv',index=False)


def evaluate_cache(c,cat,photos,cache,pre,report_root=None):
    evaluator=Evaluator()
    for p in photos:
        data=pre.load(p['path']);pack=cache.load(p['stem'],data)
        result=postprocess(pack,c,data,p['stem'],True)
        evaluator.add(p,cat.views(p['stem']),result,tuple(pack['shape']))
    if report_root:evaluator.save(report_root)
    return evaluator
''', encoding='utf-8')


(PACKAGE / 'inference.py').write_text(r'''"""Threshold-independent ROI geometry + continuous coarse conditioning permit exact cached search."""
from pathlib import Path
import importlib.metadata
import os
import time
import cv2
import numpy as np
import torch
from .config import atomic_json,digest,file_hash,autocast
from .data import square_box,roi_channels
from .preprocess import features


def tile_starts(length,size,overlap):
    if length<size:raise ValueError('Native image smaller than tile')
    points=list(range(0,length-size+1,size-overlap))
    if points[-1]!=length-size:points.append(length-size)
    return points


def smooth_window(size):
    # Positive edges ensure even outermost native pixels receive weight.
    one=np.maximum(np.hanning(size).astype(np.float32),.05)
    return one[:,None]*one[None,:]


@torch.inference_mode()
def tile_probability(model,data,c,device):
    h,w=data['raw'].shape;size=c.TILE_SIZE
    bounds=[(x,y,x+size,y+size) for y in tile_starts(h,size,c.TILE_OVERLAP) for x in tile_starts(w,size,c.TILE_OVERLAP)]
    result=np.zeros((h,w),np.float32);weights=np.zeros((h,w),np.float32);window=smooth_window(size)
    for start in range(0,len(bounds),c.INFERENCE_BATCH_SIZE):
        batch=bounds[start:start+c.INFERENCE_BATCH_SIZE]
        x=torch.from_numpy(np.stack([features(data,c.INPUT_MODE,b,size) for b in batch])).to(device)
        with autocast(c,device):p=model(x).sigmoid().float().cpu().numpy()[:,0]
        for prob,(x0,y0,x1,y1) in zip(p,batch):
            result[y0:y1,x0:x1]+=prob*window;weights[y0:y1,x0:x1]+=window
    if np.any(weights<=0):raise RuntimeError('Uncovered tile pixels')
    return result/weights


def save_npz(path,pack):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.stem+f'.{os.getpid()}.tmp.npz')
    np.savez_compressed(tmp,**pack);os.replace(tmp,path)


def read_pack(path):
    with np.load(path,allow_pickle=False) as z:return {k:z[k] for k in z.files}


@torch.inference_mode()
def detector_pack(model,data,c,device):
    h,w=data['raw'].shape;size=c.DETECTOR_IMAGE_SIZE
    tensor=torch.from_numpy(features(data,c.INPUT_MODE,size=size)).to(device)
    with autocast(c,device):out=model([tensor])[0]
    scores=out['scores'].float().cpu().numpy();boxes=out['boxes'].float().cpu().numpy()
    probs=out['masks'][:,0].float().cpu().numpy()
    boxes[:,[0,2]]*=w/size;boxes[:,[1,3]]*=h/size
    pack={'shape':np.array([h,w]),'detector_size':np.array(size),'scores':scores,'boxes':boxes,
          'pixel_hash':data['pixel_hash']}
    bounds=[]
    for i,p in enumerate(probs):
        y,x=np.nonzero(p>0)
        b=[int(x.min()),int(y.min()),int(x.max()+1),int(y.max()+1)] if len(x) else [0,0,1,1]
        x0,y0,x1,y1=b;bounds.append(b);pack[f'det_{i}']=p[y0:y1,x0:x1]
    pack['det_bounds']=np.array(bounds,np.int32).reshape(-1,4)
    return pack


def native_coarse(pack,index):
    size=int(pack['detector_size']);h,w=map(int,pack['shape'])
    full=np.zeros((size,size),np.float32);x0,y0,x1,y1=map(int,pack['det_bounds'][index])
    full[y0:y1,x0:x1]=pack[f'det_{index}']
    return cv2.resize(full,(w,h),interpolation=cv2.INTER_LINEAR)


@torch.inference_mode()
def add_refiner(pack,data,model,c,device):
    bounds=[square_box(b,data['raw'].shape,c.REFINER_CONTEXT,c.REFINER_MIN_CROP) for b in pack['boxes']]
    result=[]
    for start in range(0,len(bounds),c.INFERENCE_BATCH_SIZE):
        inputs=[]
        for i in range(start,min(start+c.INFERENCE_BATCH_SIZE,len(bounds))):
            x0,y0,x1,y1=bounds[i];coarse=native_coarse(pack,i)[y0:y1,x0:x1]
            inputs.append(roi_channels(data,bounds[i],c.REFINER_SIZE,coarse,c))
        x=torch.from_numpy(np.stack(inputs)).to(device)
        with autocast(c,device):p=model(x).sigmoid().float().cpu().numpy()[:,0]
        result.extend(p)
    pack=dict(pack);pack['ref_bounds']=np.asarray(bounds,np.int32).reshape(-1,4)
    pack['ref_probs']=np.asarray(result,np.float32).reshape(-1,c.REFINER_SIZE,c.REFINER_SIZE)
    return pack


class PredictionCache:
    def __init__(self,c,detector_path,refiner_path=None,tile_path=None,norm=None):
        source={p.name:file_hash(p) for p in Path(__file__).parent.glob('*.py')}
        provenance={'detector':file_hash(detector_path),
                    'refiner':file_hash(refiner_path) if refiner_path else None,
                    'tile':file_hash(tile_path) if tile_path else None,
                    'features':c.INPUT_MODE,'preprocess':c.PREPROCESS,'normalization':norm,
                    'size':c.DETECTOR_IMAGE_SIZE,'score_floor':c.CACHE_SCORE_FLOOR,'max_det':c.MAX_DETECTIONS,
                    'context':c.REFINER_CONTEXT,'min_crop':c.REFINER_MIN_CROP,'ref_size':c.REFINER_SIZE,
                    'tile_size':c.TILE_SIZE,'tile_overlap':c.TILE_OVERLAP,
                    'amp':c.AMP,'torch':str(torch.__version__),'opencv':cv2.__version__,
                    'torchvision':importlib.metadata.version('torchvision'),'numpy':np.__version__,
                    'cuda':torch.version.cuda,'cudnn':torch.backends.cudnn.version(),
                    'gpus':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
                    'source':source}
        self.signature=digest(provenance)
        self.root=Path(c.OUTPUT_ROOT)/'prediction_cache'/self.signature[:24]
        self.root.mkdir(parents=True,exist_ok=True)
        atomic_json(self.root/'cache_manifest.json',provenance)
        self.c=c

    def path(self,stem):return self.root/(stem+'.npz')

    def load(self,stem,data=None):
        pack=read_pack(self.path(stem))
        if data is not None and str(pack['pixel_hash'])!=str(data['pixel_hash']):raise ValueError('Cache image pixels changed')
        return pack

    def put(self,stem,pack):
        # Fail explicitly rather than silently dropping an optional branch or candidates.
        if sum(p.stat().st_size for p in self.root.parent.rglob('*.npz'))>self.c.MAX_CACHE_GB*2**30:
            raise RuntimeError('Prediction cache budget exceeded. Archive old experiment caches or increase MAX_CACHE_GB.')
        save_npz(self.path(stem),pack)


def cache_photos(c,photos,pre,cache,detector,refiner,tile,device,base_cache=None):
    times=[]
    for i,p in enumerate(photos,1):
        start=time.perf_counter();stem=p['stem'];data=pre.load(p['path'])
        if cache.path(stem).exists():cache.load(stem,data);continue
        if base_cache is not None and base_cache.path(stem).exists():pack=base_cache.load(stem,data)
        elif detector is not None:pack=detector_pack(detector,data,c,device)
        else:raise FileNotFoundError(f'Missing fixed detector cache: {stem}')
        if refiner is not None:pack=add_refiner(pack,data,refiner,c,device)
        if tile is not None:pack['tile_prob']=tile_probability(tile,data,c,device)
        cache.put(stem,pack);times.append(time.perf_counter()-start)
        print(f'Inference cache {i}/{len(photos)} {stem}: {times[-1]:.2f}s, candidates={len(pack["scores"])}',flush=True)
    return times
''', encoding='utf-8')


(PACKAGE / 'models.py').write_text(r'''"""Original v2 detector and exact original ResNet18 decoder, with explicit input adapters."""
from pathlib import Path
import math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torchvision.models import ResNet18_Weights
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2,MaskRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign
from .reference_models import CropUNet
from .config import architecture


def pretrained_state(c,weights):
    name=Path(weights.url).name
    roots=[Path(c.PRETRAINED_DIR)] if c.PRETRAINED_DIR else []
    roots += [Path(torch.hub.get_dir())/'checkpoints']
    for root in roots:
        path=root/name
        if path.exists():return torch.load(path,map_location='cpu',weights_only=True)
    # Attached Kaggle inputs are searched only by exact official weight filename.
    matches=list(Path(c.INPUT_ROOT).rglob(name)) if Path(c.INPUT_ROOT).exists() else []
    if len(matches)==1:return torch.load(matches[0],map_location='cpu',weights_only=True)
    if not c.ALLOW_WEIGHT_DOWNLOAD:raise FileNotFoundError(f'Attach official pretrained weights: {name}')
    return weights.get_state_dict(progress=True,check_hash=True)


def warm_pretrained(c):
    if not c.PRETRAINED:return
    pretrained_state(c,MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT)
    if c.ENABLE_REFINER or c.ENABLE_TILE_BRANCH:pretrained_state(c,ResNet18_Weights.DEFAULT)


def mixed_input_init(conv):
    with torch.no_grad():
        summed=conv.weight.sum(1,keepdim=True)
        conv.weight[:,0:1].copy_(summed*.5)
        conv.weight[:,1:2].copy_(summed*.5)
        conv.weight[:,2:3].zero_()


def build_detector(c,norm,pretrained=False):
    model=maskrcnn_resnet50_fpn_v2(weights=None,weights_backbone=None)
    if pretrained:model.load_state_dict(pretrained_state(c,MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT))
    box=model.roi_heads.box_predictor.cls_score.in_features
    old=model.roi_heads.mask_predictor
    model.roi_heads.box_predictor=FastRCNNPredictor(box,2)
    model.roi_heads.mask_predictor=MaskRCNNPredictor(old.conv5_mask.in_channels,old.conv5_mask.out_channels,2)
    model.rpn.anchor_generator=AnchorGenerator(sizes=tuple((s,) for s in c.ANCHOR_SIZES),aspect_ratios=(tuple(c.ANCHOR_RATIOS),)*5)
    # Preserve the original RPN conv stack when changing aspect-ratio count.
    n=len(c.ANCHOR_RATIOS);head=model.rpn.head
    if n!=3:
        cls=nn.Conv2d(256,n,1);boxlayer=nn.Conv2d(256,n*4,1)
        with torch.no_grad():
            for j,r in enumerate(c.ANCHOR_RATIOS):
                k=min(range(3),key=lambda i:abs(math.log(r/[.5,1.,2.][i])))
                cls.weight[j].copy_(head.cls_logits.weight[k]);cls.bias[j].copy_(head.cls_logits.bias[k])
                boxlayer.weight[4*j:4*j+4].copy_(head.bbox_pred.weight[4*k:4*k+4])
                boxlayer.bias[4*j:4*j+4].copy_(head.bbox_pred.bias[4*k:4*k+4])
        head.cls_logits,head.bbox_pred=cls,boxlayer
    model.roi_heads.mask_roi_pool=MultiScaleRoIAlign(['0','1','2','3'],28,2)
    model.transform.min_size=(c.DETECTOR_IMAGE_SIZE,);model.transform.max_size=c.DETECTOR_IMAGE_SIZE
    model.transform.image_mean=norm['mean'];model.transform.image_std=norm['std']
    model.roi_heads.score_thresh=c.CACHE_SCORE_FLOOR;model.roi_heads.detections_per_img=c.MAX_DETECTIONS
    # Match pretrained torchvision's three trainable ResNet stages. Mixed inputs
    # additionally train conv1 so its initially-zero radial weights can learn.
    for name,p in model.backbone.body.named_parameters():
        p.requires_grad_(name.startswith(('layer2','layer3','layer4')) or
                         (c.INPUT_MODE=='raw_corrected_radial' and name.startswith('conv1')))
    if c.INPUT_MODE=='raw_corrected_radial' and pretrained:mixed_input_init(model.backbone.body.conv1)
    return model


class Segmenter(CropUNet):
    def __init__(self,c,norm,stage,pretrained=False):
        # Exact reference ResNet18 encoder and transposed-convolution decoder.
        super().__init__(pretrained=False)
        if pretrained:
            state=pretrained_state(c,ResNet18_Weights.DEFAULT)
            self.stem[0].weight.data.copy_(state['conv1.weight'])
            self.stem[1].load_state_dict({k.removeprefix('bn1.'):v for k,v in state.items() if k.startswith('bn1.')})
            for name in ['layer1','layer2','layer3','layer4']:
                getattr(self,name).load_state_dict({k[len(name)+1:]:v for k,v in state.items() if k.startswith(name+'.')})
        if stage=='refiner':
            n=3+int(c.REFINER_USE_COARSE)+int(c.REFINER_USE_RADIAL)
            old=self.stem[0]
            conv=nn.Conv2d(n,64,7,2,3,bias=False)
            with torch.no_grad():conv.weight.zero_();conv.weight[:,:3].copy_(old.weight)
            self.stem[0]=conv
            means=[.485,.456,.406]+[.5]*(n-3);stds=[.229,.224,.225]+[.25]*(n-3)
        else:
            means=norm['mean'];stds=norm['std']
            if c.INPUT_MODE=='raw_corrected_radial' and pretrained:mixed_input_init(self.stem[0])
        self.mean=torch.tensor(means).view(1,-1,1,1)
        self.std=torch.tensor(stds).view(1,-1,1,1)


def bce_tversky(logits,target,alpha=.3,beta=.7):
    # Accumulate small structures in float32 even inside AMP.
    logits=logits.float();target=target.float();p=logits.sigmoid()
    dims=(1,2,3);tp=(p*target).sum(dims);fp=(p*(1-target)).sum(dims);fn=((1-p)*target).sum(dims)
    return .5*F.binary_cross_entropy_with_logits(logits,target)+.5*(1-(tp+1)/(tp+alpha*fp+beta*fn+1)).mean()


def checkpoint_path(c,stage,kind='best',all_data=False):
    root=Path(c.CHECKPOINT_DIR) if c.CHECKPOINT_DIR else Path(c.OUTPUT_ROOT)/'checkpoints'
    return root/(('all_' if all_data else '')+stage+'_'+kind+'.pt')


def load_model(c,norm,stage,device,path=None):
    path=Path(path) if path else checkpoint_path(c,stage)
    state=torch.load(path,map_location='cpu',weights_only=True)
    if state['architecture']!=architecture(c,stage):raise ValueError(f'Checkpoint architecture/config mismatch: {path}')
    if any(not np.allclose(state['normalization'][k],norm[k],rtol=1e-6,atol=1e-7) for k in ['mean','std']):
        raise ValueError(f'Checkpoint training normalization differs from current pipeline: {path}')
    model=build_detector(c,norm) if stage=='detector' else Segmenter(c,norm,stage)
    model.load_state_dict(state['model'],strict=True)
    return model.to(device).eval(),state
''', encoding='utf-8')


(PACKAGE / 'postprocess.py').write_text(r'''from dataclasses import dataclass,replace
from pathlib import Path
import json
import cv2
import numpy as np
from pycocotools import mask as cm
from .inference import native_coarse
from .pq import canonical_rle,pairwise_iou


@dataclass
class Instance:
    score: float
    rle: dict
    quality: float = 1.
    source: str = 'global'


def encode(mask):return cm.encode(np.asfortranarray(mask.astype(np.uint8)))


def mask_nms(instances,threshold,soft_sigma=None,min_score=0.):
    pool=sorted(instances,key=lambda x:-x.score);result=[]
    while pool:
        best=pool.pop(0)
        if best.score<min_score:continue
        result.append(best)
        if not pool:break
        ious=pairwise_iou([best.rle],[x.rle for x in pool])[0]
        if soft_sigma is None:pool=[x for x,iou in zip(pool,ious) if iou<=threshold]
        else:
            pool=[replace(x,score=x.score*float(np.exp(-iou*iou/soft_sigma))) if iou>threshold else x for x,iou in zip(pool,ious)]
            pool.sort(key=lambda x:-x.score)
    return result


def components(mask,min_area,split=False):
    n,labels,stats,_=cv2.connectedComponentsWithStats(mask.astype(np.uint8),8)
    kept=[labels==i for i in range(1,n) if stats[i,cv2.CC_STAT_AREA]>=min_area]
    if split:return kept
    return [np.logical_or.reduce(kept)] if kept else []


def raw_instances(pack,c,disk=None):
    h,w=map(int,pack['shape']);instances=[]
    tile=pack.get('tile_prob') if c.ENABLE_TILE_BRANCH else None
    for i,score in enumerate(pack['scores']):
        if score<c.DETECTION_THRESHOLD:continue
        coarse=native_coarse(pack,i)
        if c.ENABLE_REFINER:
            if 'ref_probs' not in pack:raise ValueError('Refiner enabled but probability cache has no refiner output')
            x0,y0,x1,y1=map(int,pack['ref_bounds'][i])
            local=cv2.resize(pack['ref_probs'][i],(x1-x0,y1-y0),interpolation=cv2.INTER_LINEAR)
            prob=np.zeros((h,w),np.float32);prob[y0:y1,x0:x1]=local
            threshold=c.REFINER_THRESHOLD
        else:prob=coarse.copy();threshold=c.MASK_THRESHOLD
        if tile is not None:
            support=coarse>=c.MASK_THRESHOLD
            if c.TILE_GROW:support=cv2.dilate(support.astype(np.uint8),np.ones((2*c.TILE_GROW+1,)*2,np.uint8)).astype(bool)
            prob[support]=(1-c.TILE_FUSION_WEIGHT)*prob[support]+c.TILE_FUSION_WEIGHT*tile[support]
        mask=prob>=threshold
        if not mask.any() and c.ENABLE_REFINER and c.EMPTY_REFINER_FALLBACK:mask=coarse>=c.MASK_THRESHOLD
        if disk is not None:mask &= disk
        if c.MORPH_CLOSE_RADIUS:
            r=c.MORPH_CLOSE_RADIUS;mask=cv2.morphologyEx(mask.astype(np.uint8),cv2.MORPH_CLOSE,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*r+1,2*r+1))).astype(bool)
            if disk is not None:mask &= disk
        if not mask.any():continue
        quality=float(prob[mask].mean())
        s=float(score)*((1-c.QUALITY_WEIGHT)+c.QUALITY_WEIGHT*quality)
        instances.append(Instance(s,encode(mask),quality))
    return instances


def fuse_instances(views,weights=None,match_iou=.55):
    """Seed-based instance matching, at most one vote/model; no overlap transitive union."""
    if weights is None:weights=np.ones(len(views))
    weights=np.asarray(weights,float)
    if len(weights)!=len(views) or np.any(weights<=0):raise ValueError('One positive ensemble weight per view required')
    pool=sorted([(ins,model) for model,view in enumerate(views) for ins in view],key=lambda z:-z[0].score)
    used=set();result=[]
    for i,(seed,model) in enumerate(pool):
        if i in used:continue
        used.add(i);group=[i];seen={model}
        for j,(other,m) in enumerate(pool):
            if j in used or m in seen:continue
            if pairwise_iou([seed.rle],[other.rle])[0,0]>match_iou:used.add(j);group.append(j);seen.add(m)
        total=sum(weights[pool[j][1]]*pool[j][0].score for j in group)
        canvas=np.zeros(seed.rle['size'],np.float32);available=0.
        for j in group:
            instance,m=pool[j];a=weights[m]*instance.score
            canvas+=a*cm.decode(instance.rle);available+=a
        mask=canvas>=.5*available
        if mask.any():result.append(Instance(float(total/weights.sum()),encode(mask),source='ensemble'))
    return result


def external_instances(root,stem,validation=False):
    root=Path(root);meta=json.loads((root/'ensemble_manifest.json').read_text())
    if validation and stem in set(meta['trained_photo_ids']):raise ValueError('External ensemble predictions have training leakage')
    payload=json.loads((root/(stem+'.json')).read_text())
    if payload['photo']!=stem:raise ValueError('External prediction photo mismatch')
    return [Instance(float(x['score']),canonical_rle(x['rle']),float(x.get('quality',1)),str(root)) for x in payload['instances']]


def finalize(instances,c,shape,tile=None,disk=None):
    if c.POSTPROCESS_MODE=='legacy_union':
        # Explicit historical control only.
        adjacency=pairwise_iou([x.rle for x in instances],[x.rle for x in instances])>0 if instances else np.zeros((0,0),bool)
        seen=set();merged=[]
        for i in range(len(instances)):
            if i in seen:continue
            stack=[i];group=[]
            while stack:
                j=stack.pop()
                if j in seen:continue
                seen.add(j);group.append(j);stack.extend(np.flatnonzero(adjacency[j]).tolist())
            merged.append(Instance(max(instances[j].score for j in group),cm.merge([instances[j].rle for j in group])))
        instances=merged
    else:instances=mask_nms(instances,c.MASK_NMS_IOU,c.SOFT_NMS_SIGMA if c.POSTPROCESS_MODE=='soft_nms' else None,c.DETECTION_THRESHOLD)
    occupied=np.zeros(shape,bool);result=[]
    for ins in sorted(instances,key=lambda x:-x.score):
        mask=cm.decode(ins.rle).astype(bool)&~occupied
        if disk is not None:mask &= disk
        masks=components(mask,c.FRAGMENT_MIN_AREA,c.FRAGMENT_MODE=='split') if c.FRAGMENT_MODE!='keep' else [mask]
        for part in masks:
            if part.sum()<c.MIN_INSTANCE_AREA:continue
            occupied|=part;result.append(replace(ins,rle=encode(part)))
    if tile is not None and c.TILE_KEEP_UNCLAIMED:
        mask=(tile>=c.TILE_THRESHOLD)&~occupied
        if disk is not None:mask &= disk
        for part in components(mask,c.TILE_UNCLAIMED_MIN_AREA,True):result.append(Instance(c.TILE_THRESHOLD,encode(part),source='tile_unclaimed'))
    return sorted(result,key=lambda x:(float(cm.toBbox(x.rle)[1]),float(cm.toBbox(x.rle)[0])))


def postprocess(pack,c,data=None,stem=None,validation=False):
    shape=tuple(map(int,pack['shape']));disk=None
    if data is not None and c.PREPROCESS:
        yy,xx=np.ogrid[:shape[0],:shape[1]];cx,cy,r=data['disk']
        disk=(xx-cx)**2+(yy-cy)**2<=(r*c.DISK_MARGIN)**2
    own=raw_instances(pack,c,disk)
    if c.ENSEMBLE_CACHE_DIRS:
        views=[own]+[external_instances(root,stem,validation) for root in c.ENSEMBLE_CACHE_DIRS]
        own=fuse_instances(views,c.ENSEMBLE_WEIGHTS or None,c.ENSEMBLE_MATCH_IOU)
    return finalize(own,c,shape,pack.get('tile_prob') if c.ENABLE_TILE_BRANCH else None,disk)
''', encoding='utf-8')


(PACKAGE / 'pq.py').write_text(r'''"""Official-notebook-compatible pooled PQ. No model or dataset downloads.

Reference: azimahmadzadeh/self-evaluation-notebook, version 6, retrieved
2026-11-02. The reference counts ALL IoU > 0.5 pairs, not Hungarian matches.
This module deliberately preserves that behavior, including overlapping GT.
"""
from __future__ import annotations

from dataclasses import asdict, dataclass
from typing import Mapping, Sequence

import numpy as np
from pycocotools import mask as coco_mask


@dataclass
class Totals:
    tp: int = 0
    fp: int = 0
    fn: int = 0
    iou_sum: float = 0.0
    ambiguous_gt: int = 0
    ambiguous_pred: int = 0
    fragmented_gt: int = 0
    merged_pred: int = 0

    def __add__(self, other: Totals) -> Totals:
        return Totals(**{k: v + asdict(other)[k] for k, v in asdict(self).items()})

    def summary(self) -> dict:
        denominator = self.tp + 0.5 * self.fp + 0.5 * self.fn
        return {
            **asdict(self),
            "pq": self.iou_sum / denominator if denominator else 0.0,
            "dq": self.tp / denominator if denominator else 0.0,
            "sq": self.iou_sum / self.tp if self.tp else 0.0,
        }


def from_iou(ious: np.ndarray) -> Totals:
    """Count exactly as the host notebook; diagnostics use any positive IoU."""
    ious = np.asarray(ious, dtype=np.float64)
    if ious.ndim != 2 or not np.isfinite(ious).all():
        raise ValueError("IoU matrix must be a finite 2-D array")
    if ((ious < 0) | (ious > 1)).any():
        raise ValueError("IoU values must be in [0, 1]")
    hit = ious > 0.5
    links = ious > 0
    return Totals(
        tp=int(hit.sum()),
        fp=int((hit.sum(axis=0) == 0).sum()),
        fn=int((hit.sum(axis=1) == 0).sum()),
        iou_sum=float(ious[hit].sum()),
        ambiguous_gt=int((hit.sum(axis=1) > 1).sum()),
        ambiguous_pred=int((hit.sum(axis=0) > 1).sum()),
        fragmented_gt=int((links.sum(axis=1) > 1).sum()),
        merged_pred=int((links.sum(axis=0) > 1).sum()),
    )


def canonical_rle(rle: dict) -> dict:
    size = list(map(int, rle["size"]))
    if len(size) != 2 or min(size) <= 0:
        raise ValueError("RLE size must be positive height,width")
    counts = rle["counts"]
    if isinstance(counts, str):
        counts = counts.encode("ascii")
    if isinstance(counts, list):
        return coco_mask.frPyObjects({"size": size, "counts": counts}, *size)
    return {"size": size, "counts": counts}


def pairwise_iou(gt: Sequence[dict], pred: Sequence[dict]) -> np.ndarray:
    if not gt or not pred:
        return np.zeros((len(gt), len(pred)), dtype=np.float64)
    gt, pred = list(map(canonical_rle, gt)), list(map(canonical_rle, pred))
    shapes = {tuple(r["size"]) for r in gt + pred}
    if len(shapes) != 1:
        raise ValueError("GT and prediction RLE sizes differ; resize before encoding")
    return np.asarray(coco_mask.iou(pred, gt, [0] * len(gt)), dtype=np.float64).T


def require_disjoint(pred: Sequence[dict], shape=(2048, 2048)) -> None:
    """Reject invalid submissions rather than silently changing their masks."""
    occupied = np.zeros(shape, dtype=bool)
    for r in pred:
        r = canonical_rle(r)
        if tuple(r["size"]) != tuple(shape):
            raise ValueError("Prediction has unexpected canvas size")
        current = coco_mask.decode(r).astype(bool)
        if not current.any():
            raise ValueError("Empty mask is not a filament instance")
        if (occupied & current).any():
            raise ValueError("Prediction instances overlap")
        occupied |= current


def evaluate_records(
    records: Mapping[str, tuple[str, Sequence[dict]]],
    predictions: Mapping[str, Sequence[dict]],
    *, validate_predictions: bool = True, shape=(2048, 2048),
) -> dict:
    """One prediction set per photo, scored against EVERY annotator separately.

    records: {annotator_record_id: (physical_photo_stem, list_of_GT_RLEs)}.
    Missing predictions are [], and unknown prediction photos are rejected.
    Explicit zero-GT records are supported using the host score's empty-row
    rule. Unlike its CSV-only entry discovery, the manifest retains such rows.
    """
    known = {stem for stem, _ in records.values()}
    unknown = set(predictions) - known
    if unknown:
        raise ValueError(f"Predictions outside evaluation split: {sorted(unknown)[:3]}")
    for stem in known:
        if validate_predictions:
            require_disjoint(predictions.get(stem, []), shape)
    total = Totals()
    by_photo: dict[str, Totals] = {}
    by_record = {}
    nonzero_ious, matched_ious = [], []
    for record_id, (stem, gt) in records.items():
        pred = predictions.get(stem, [])
        for r in list(gt) + list(pred):
            if tuple(r["size"]) != tuple(shape):
                raise ValueError("RLE canvas differs from evaluation canvas")
        ious = pairwise_iou(gt, pred)
        result = from_iou(ious)
        total = total + result
        by_photo[stem] = by_photo.get(stem, Totals()) + result
        by_record[record_id] = result.summary()
        nonzero_ious.extend(ious[ious > 0].tolist())
        matched_ious.extend(ious[ious > 0.5].tolist())
    return {
        "pooled": total.summary(),
        "n_photos": len(known), "n_annotator_records": len(records),
        "n_prediction_instances_physical": sum(len(predictions.get(s, [])) for s in known),
        "per_photo": {s: t.summary() for s, t in sorted(by_photo.items())},
        "per_record": by_record,
        "positive_pair_ious": nonzero_ious,
        "positive_pair_dices": [2 * x / (1 + x) for x in nonzero_ious],
        "matched_pair_ious": matched_ious,
        "notes": "Pooled all-pairs host semantics; no one-to-one assignment. Empty total returns 0.",
    }
''', encoding='utf-8')


(PACKAGE / 'preprocess.py').write_text(r'''"""Disk fitting is adapted from ameyypawar/filament-seg (MIT; license bundled)."""
from functools import lru_cache
from pathlib import Path
import hashlib
import os
import cv2
import numpy as np
from .config import digest
from .disk import detect_disk


def read_gray(path):
    image=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE)
    if image is None:raise FileNotFoundError(path)
    return image


class SolarPreprocessor:
    VERSION='solar-v1-radial-median-edgepad-2'

    def __init__(self,root,enabled=True):
        self.root=Path(root);self.root.mkdir(parents=True,exist_ok=True)
        self.enabled=enabled

    def process(self,image):
        disk=detect_disk(image)
        radial=disk.radius_map(image.shape).astype(np.float32)
        inside=radial<=1.
        profile=np.full(128,np.nan,np.float32)
        # Sample every second pixel: robust radial statistics without a large sort.
        r=radial[::2,::2];v=image[::2,::2].astype(np.float32)
        bins=np.minimum((r*128).astype(int),127)
        for b in range(128):
            values=v[(bins==b)&(r<=1.)]
            if len(values):profile[b]=np.median(values)
        valid=np.isfinite(profile)&(profile>0)
        if valid.any():profile=np.interp(np.arange(128),np.flatnonzero(valid),profile[valid]).astype(np.float32)
        else:profile.fill(max(float(image.mean()),1.))
        profile=np.convolve(np.pad(profile,(4,4),mode='edge'),np.ones(9)/9,mode='valid').astype(np.float32)
        background=np.interp(np.clip(radial*127,0,127),np.arange(128),profile).astype(np.float32)
        corrected=image.astype(np.float32)/np.maximum(background,1.)
        values=corrected[inside]
        lo,hi=np.percentile(values,[1,99.5]) if len(values) else (0.,1.)
        corrected=np.clip((corrected-lo)/max(float(hi-lo),.05),0,1)
        corrected[~inside]=0
        if not self.enabled:corrected=image.astype(np.float32)/255.
        return {'raw':image,'corrected':np.rint(corrected*255).astype(np.uint8),'disk_mask':inside.astype(np.uint8),
                'disk':np.array([disk.cx,disk.cy,disk.radius],np.float64),'profile':profile,
                'pixel_hash':np.asarray(hashlib.sha256(image.tobytes()).hexdigest())}

    def path(self,path):
        p=Path(path);s=p.stat()
        key=digest([self.VERSION,str(p.resolve()),s.st_size,s.st_mtime_ns,self.enabled])[:24]
        return self.root/(p.stem+'_'+key+'.npz')

    def ensure(self,path):
        dest=self.path(path)
        if not dest.exists():
            data=self.process(read_gray(path))
            tmp=dest.with_name(dest.stem+f'.{os.getpid()}.tmp.npz')
            np.savez_compressed(tmp,**data);os.replace(tmp,dest)
        return dest

    @lru_cache(maxsize=2)
    def load(self,path):
        with np.load(self.ensure(path),allow_pickle=False) as z:
            return {k:z[k] for k in z.files}


def radial_window(data,bounds=None):
    h,w=data['raw'].shape
    x0,y0,x1,y1=(0,0,w,h) if bounds is None else map(int,bounds)
    cx,cy,r=data['disk']
    yy,xx=np.ogrid[y0:y1,x0:x1]
    return np.clip(np.hypot(xx-cx,yy-cy)/max(r,1.),0,1).astype(np.float32)


def features(data,mode,bounds=None,size=None):
    h,w=data['raw'].shape
    x0,y0,x1,y1=(0,0,w,h) if bounds is None else map(int,bounds)
    raw=data['raw'][y0:y1,x0:x1].astype(np.float32)/255.
    corrected=data['corrected'][y0:y1,x0:x1].astype(np.float32)/255.
    if mode=='raw_repeat':array=np.repeat(raw[...,None],3,axis=2)
    elif mode=='corrected_repeat':array=np.repeat(corrected[...,None],3,axis=2)
    elif mode=='raw_corrected_radial':array=np.stack([raw,corrected,radial_window(data,bounds)],axis=-1)
    else:raise ValueError(mode)
    if size is not None:
        interp=cv2.INTER_AREA if min(array.shape[:2])>size else cv2.INTER_LINEAR
        array=cv2.resize(array,(size,size),interpolation=interp)
    return np.ascontiguousarray(array.transpose(2,0,1))


def fit_normalization(c,preprocessor,photos):
    if c.INPUT_MODE!='raw_corrected_radial':
        return {'mean':[.485,.456,.406],'std':[.229,.224,.225],'source':'ImageNet RGB convention'}
    total=np.zeros(3);squared=np.zeros(3);n=0
    for p in photos:
        f=features(preprocessor.load(p['path']),c.INPUT_MODE,size=128).reshape(3,-1).astype(np.float64)
        total+=f.sum(1);squared+=(f*f).sum(1);n+=f.shape[1]
    mean=total/n;std=np.sqrt(np.maximum(squared/n-mean**2,1e-4))
    return {'mean':mean.tolist(),'std':std.tolist(),'source':'training physical photos only; full-frame pixels'}
''', encoding='utf-8')


(PACKAGE / 'reference_models.py').write_text(r'''"""Architectures extracted unchanged from Nomannic19's public reference notebook.
Source: https://www.kaggle.com/code/nomannic19/cv-solar-filament-mask-r-cnn-u-net-refiner
Reference notebook is Apache-2.0. No pretrained weights are downloaded by this module.
Only class/build/crop definitions are included; no training code is executed.
"""
import numpy as np
import torch
from torch import nn
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign
IMAGE_SIZE = 1024

def build_model(pretrained=True):
    weights = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT if pretrained else None
    model = maskrcnn_resnet50_fpn_v2(weights=weights, weights_backbone=None)

    box_features = model.roi_heads.box_predictor.cls_score.in_features
    mask_features = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden = model.roi_heads.mask_predictor.conv5_mask.out_channels
    model.roi_heads.box_predictor = FastRCNNPredictor(box_features, 2)
    model.roi_heads.mask_predictor = MaskRCNNPredictor(mask_features, hidden, 2)

    model.rpn.anchor_generator = AnchorGenerator(sizes=((16,), (32,), (64,), (128,), (256,)), aspect_ratios=((0.5, 1.0, 2.0),) * 5)
    model.roi_heads.mask_roi_pool = MultiScaleRoIAlign(featmap_names=["0", "1", "2", "3"], output_size=28, sampling_ratio=2)
    model.transform.min_size, model.transform.max_size = (IMAGE_SIZE,), IMAGE_SIZE
    return model

def square_bounds(mask, context=1.8, minimum=96):
    height, width = mask.shape
    ys, xs = np.where(mask)
    center_x, center_y = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2
    side = int(np.ceil(max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * context))
    side = min(max(side, minimum), min(height, width))
    x0, y0 = int(round(center_x - side / 2)), int(round(center_y - side / 2))
    x1, y1 = x0 + side, y0 + side

    if x0 < 0:
        x1 -= x0; x0 = 0
    if y0 < 0:
        y1 -= y0; y0 = 0
    if x1 > width:
        x0 -= x1 - width; x1 = width
    if y1 > height:
        y0 -= y1 - height; y1 = height

    return int(x0), int(y0), int(x1), int(y1)

class ConvBlock(nn.Sequential):
    def __init__(self, in_channels, out_channels):
        super().__init__(nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True), nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True))

class CropUNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        encoder = resnet18(weights=ResNet18_Weights.DEFAULT if pretrained else None)
        self.stem = nn.Sequential(encoder.conv1, encoder.bn1, encoder.relu)
        self.pool, self.layer1, self.layer2, self.layer3, self.layer4 = encoder.maxpool, encoder.layer1, encoder.layer2, encoder.layer3, encoder.layer4
        self.up4, self.dec4 = nn.ConvTranspose2d(512, 256, 2, 2), ConvBlock(512, 256)
        self.up3, self.dec3 = nn.ConvTranspose2d(256, 128, 2, 2), ConvBlock(256, 128)
        self.up2, self.dec2 = nn.ConvTranspose2d(128, 64, 2, 2), ConvBlock(128, 64)
        self.up1, self.dec1 = nn.ConvTranspose2d(64, 32, 2, 2), ConvBlock(96, 32)
        self.up0, self.head = nn.ConvTranspose2d(32, 16, 2, 2), nn.Conv2d(16, 1, 1)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, image):
        image = (image - self.mean) / self.std
        x0 = self.stem(image)
        x1 = self.layer1(self.pool(x0))
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        x = self.dec4(torch.cat([self.up4(x4), x3], 1))
        x = self.dec3(torch.cat([self.up3(x), x2], 1))
        x = self.dec2(torch.cat([self.up2(x), x1], 1))
        x = self.dec1(torch.cat([self.up1(x), x0], 1))
        return self.head(self.up0(x))
''', encoding='utf-8')


(PACKAGE / 'submission.py').write_text(r'''from pathlib import Path
import pandas as pd
from pycocotools import mask as cm
from .config import atomic_json
from .pq import require_disjoint


def export_instances(root,stem,instances):
    rows=[]
    for x in instances:
        r=x.rle;counts=r['counts'].decode('ascii') if isinstance(r['counts'],bytes) else r['counts']
        rows.append({'score':float(x.score),'quality':float(x.quality),'rle':{'size':list(map(int,r['size'])),'counts':counts}})
    atomic_json(Path(root)/(stem+'.json'),{'photo':stem,'instances':rows})


class SubmissionWriter:
    def __init__(self,path,test_ids,shape=(2048,2048),coverage_path=None):
        self.path=Path(path);self.ids=set(test_ids);self.shape=shape
        if len(self.ids)!=len(test_ids):raise ValueError('Duplicate test observations')
        self.coverage_path=Path(coverage_path) if coverage_path else self.path.with_name('submission_coverage.json')
        self.rows=[];self.coverage={}

    def add(self,stem,instances):
        if stem not in self.ids or stem in self.coverage:raise ValueError('Unknown or repeated test photograph')
        rles=[x.rle for x in instances];require_disjoint(rles,self.shape)
        for number,r in enumerate(rles,1):
            counts=r['counts'].decode('ascii') if isinstance(r['counts'],bytes) else r['counts']
            # Decode check also rejects malformed/truncated COCO RLEs.
            mask=cm.decode({'size':list(self.shape),'counts':counts.encode('ascii')})
            if mask.shape!=self.shape or not mask.any():raise ValueError('Invalid submission mask')
            self.rows.append({'filament_id':f'{stem}_{number}','segmentation_rle':counts})
        # Zero-instance photos are represented in coverage, not by fabricated CSV masks.
        self.coverage[stem]=len(rles)

    def finish(self):
        if set(self.coverage)!=self.ids:raise ValueError(f'Unprocessed test photographs: {sorted(self.ids-set(self.coverage))[:5]}')
        frame=pd.DataFrame(self.rows,columns=['filament_id','segmentation_rle'])
        if not frame.filament_id.is_unique:raise ValueError('Duplicate filament_id')
        self.path.parent.mkdir(parents=True,exist_ok=True);frame.to_csv(self.path,index=False)
        atomic_json(self.coverage_path,{'all_test_photos_processed':True,'n_photos':len(self.ids),
            'n_instances':len(frame),'empty_photos':[s for s,n in self.coverage.items() if n==0],
            'counts_by_photo':self.coverage,'shape':list(self.shape)})
        return frame
''', encoding='utf-8')


(PACKAGE / 'train.py').write_text(r'''"""DDP/AMP training. Every best checkpoint is selected with native end-to-end official PQ."""
import argparse
from contextlib import nullcontext
from dataclasses import replace
from pathlib import Path
import json
import os
import time
import numpy as np
import pandas as pd
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
from .config import Config,architecture,digest,seed_all,autocast,file_hash
from .data import Catalog,DetectorDataset,SegDataset,detection_collate,load_manifest
from .preprocess import SolarPreprocessor
from .models import build_detector,Segmenter,bce_tversky,checkpoint_path
from .inference import detector_pack,add_refiner,tile_probability,PredictionCache
from .postprocess import postprocess
from .evaluation import Evaluator


def atomic_torch(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp');torch.save(obj,tmp);os.replace(tmp,path)


def seed_worker(worker_id):
    torch.set_num_threads(1)
    np.random.seed(torch.initial_seed()%2**32)


def train_epoch(model,loader,optimizer,scaler,c,stage,device,accum):
    model.train();optimizer.zero_grad(set_to_none=True);loss_total=0.;samples=0
    for step,batch in enumerate(loader):
        boundary=(step+1)%accum==0 or step+1==len(loader)
        divisor=min(accum,len(loader)-(step//accum)*accum)
        sync=nullcontext() if boundary or not isinstance(model,DDP) else model.no_sync()
        with sync:
            if stage=='detector':
                images,targets,weights=batch
                if len(images)!=1:raise ValueError('Per-record physical balancing requires detector batch size 1 per GPU')
                images=[x.to(device,non_blocking=True) for x in images]
                targets=[{k:v.to(device,non_blocking=True) for k,v in t.items()} for t in targets]
                with autocast(c,device):loss=sum(model(images,targets).values())*weights[0]
                n=1
            else:
                x,y=batch;x=x.to(device,non_blocking=True);y=y.to(device,non_blocking=True)
                with autocast(c,device):loss=bce_tversky(model(x),y,c.TVERSKY_ALPHA,c.TVERSKY_BETA)
                n=len(x)
            if not torch.isfinite(loss):raise FloatingPointError(f'Nonfinite {stage} loss')
            scaler.scale(loss/divisor).backward()
        if boundary:
            scaler.unscale_(optimizer);torch.nn.utils.clip_grad_norm_(model.parameters(),10.)
            scaler.step(optimizer);scaler.update();optimizer.zero_grad(set_to_none=True)
        loss_total+=float(loss.detach())*n;samples+=n
    totals=torch.tensor([loss_total,samples],device=device,dtype=torch.float64)
    if dist.is_initialized():dist.all_reduce(totals)
    return float(totals[0]/totals[1])


@torch.inference_mode()
def validate_epoch(model,c,cat,photos,pre,norm,stage,device,base_cache=None,report_root=None):
    model.eval();ev=Evaluator()
    trial=replace(c,ENABLE_REFINER=stage!='detector' and c.ENABLE_REFINER,ENABLE_TILE_BRANCH=stage=='tile')
    for p in photos:
        data=pre.load(p['path'])
        if stage=='detector':pack=detector_pack(model,data,c,device)
        else:
            pack=base_cache.load(p['stem'],data)
            if stage=='refiner':pack=add_refiner(pack,data,model,c,device)
            else:pack={**pack,'tile_prob':tile_probability(model,data,c,device)}
        instances=postprocess(pack,trial,data,p['stem'],True)
        ev.add(p,cat.views(p['stem']),instances,tuple(data['raw'].shape))
    if report_root:ev.save(report_root)
    return ev.summary()


def rng_state(device):
    return {'cpu':torch.get_rng_state(),'cuda':torch.cuda.get_rng_state(device) if device.type=='cuda' else None}


def restore_rng(value,device):
    torch.set_rng_state(value['cpu'])
    if device.type=='cuda' and value['cuda'] is not None:torch.cuda.set_rng_state(value['cuda'],device)


def train(c,stage,all_data=False,epochs_override=None):
    import cv2
    if not torch.cuda.is_available():raise RuntimeError('Real training requires a Kaggle GPU; CPU is reserved for synthetic verification.')
    world=int(os.environ.get('WORLD_SIZE','1'));rank=int(os.environ.get('RANK','0'));local=int(os.environ.get('LOCAL_RANK','0'))
    torch.cuda.set_device(local);device=torch.device('cuda',local)
    if world>1:dist.init_process_group('nccl',timeout=__import__('datetime').timedelta(hours=2))
    seed_all(c.SEED+rank);torch.set_num_threads(2);cv2.setNumThreads(0)
    torch.backends.cudnn.benchmark=True
    cat=Catalog(c);manifest=load_manifest(c);norm=json.loads((Path(c.OUTPUT_ROOT)/'normalization.json').read_text())
    if cat.hash!=manifest['annotation_hash']:raise ValueError('Training annotations changed since split creation')
    train_photos=[p for p in manifest['photos'] if all_data or p['split']=='train']
    val_photos=[p for p in manifest['photos'] if p['split']=='val']
    dataset=DetectorDataset(c,cat,train_photos) if stage=='detector' else SegDataset(c,cat,train_photos,stage)
    sampler=DistributedSampler(dataset,num_replicas=world,rank=rank,shuffle=True,seed=c.SEED)
    batch_size=1 if stage=='detector' else c.REFINER_BATCH_SIZE if stage=='refiner' else c.TILE_BATCH_SIZE
    accum=c.DETECTOR_ACCUM_STEPS if stage=='detector' else c.REFINER_ACCUM_STEPS if stage=='refiner' else c.TILE_ACCUM_STEPS
    generator=torch.Generator()
    loader=DataLoader(dataset,batch_size=batch_size,sampler=sampler,num_workers=c.NUM_WORKERS,
        pin_memory=True,persistent_workers=c.NUM_WORKERS>0,worker_init_fn=seed_worker,generator=generator,
        collate_fn=detection_collate if stage=='detector' else None)
    epochs=epochs_override or (c.DETECTOR_EPOCHS if stage=='detector' else c.REFINER_EPOCHS if stage=='refiner' else c.TILE_EPOCHS)
    model=(build_detector(c,norm,c.PRETRAINED) if stage=='detector' else Segmenter(c,norm,stage,c.PRETRAINED)).to(device)
    if world>1:model=DDP(model,device_ids=[local],broadcast_buffers=False,find_unused_parameters=stage=='detector')
    module=model.module if isinstance(model,DDP) else model
    if stage=='detector':
        optimizer=torch.optim.SGD([p for p in model.parameters() if p.requires_grad],lr=c.DETECTOR_LR,momentum=.9,weight_decay=1e-4)
        scheduler=torch.optim.lr_scheduler.MultiStepLR(optimizer,sorted(set([max(1,round(epochs*.7)),max(2,round(epochs*.9))])),gamma=.1)
    else:
        optimizer=torch.optim.AdamW(model.parameters(),lr=c.SEGMENTER_LR,weight_decay=1e-4)
        scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=epochs)
    scaler=torch.amp.GradScaler('cuda',enabled=c.AMP)
    signature=digest({'architecture':architecture(c,stage),'split':manifest['split_hash'],'all_data':all_data,
        'epochs':epochs,'world_size':world,'batch_size':batch_size,'accum':accum,'annotation_mode':c.ANNOTATION_MODE,
        'seed':c.SEED,'normalization':norm,'lr':c.DETECTOR_LR if stage=='detector' else c.SEGMENTER_LR,
        'refiner_context':c.REFINER_CONTEXT,'refiner_jitter':c.REFINER_BOX_JITTER,
        'tversky':[c.TVERSKY_ALPHA,c.TVERSKY_BETA],
        'selection_config':{k:getattr(c,k) for k in ['DETECTION_THRESHOLD','MASK_THRESHOLD','REFINER_THRESHOLD',
            'MASK_NMS_IOU','MIN_INSTANCE_AREA','POSTPROCESS_MODE','DISK_MARGIN','MORPH_CLOSE_RADIUS','FRAGMENT_MODE']},
        'source':{p.name:file_hash(p) for p in Path(__file__).parent.glob('*.py')}})
    last=checkpoint_path(c,stage,'last',all_data);best=checkpoint_path(c,stage,'best',all_data)
    start=1;best_pq=-1.;history=[]
    if c.RESUME and last.exists():
        state=torch.load(last,map_location='cpu',weights_only=True)
        if state['training_signature']!=signature:raise ValueError('Resume training configuration mismatch; use a new OUTPUT_ROOT')
        module.load_state_dict(state['model']);optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler']);scaler.load_state_dict(state['scaler'])
        start=state['epoch']+1;best_pq=state['best_pq'];history=state['history']
        restore_rng(state['rng'][rank],device)
    base=None;pre=SolarPreprocessor(Path(c.OUTPUT_ROOT)/'preprocessed',c.PREPROCESS)
    if rank==0 and stage!='detector' and not all_data:
        det=checkpoint_path(c,'detector')
        ref=checkpoint_path(c,'refiner') if stage=='tile' and c.ENABLE_REFINER else None
        base=PredictionCache(c,det,ref,norm=norm)
        missing=[p['stem'] for p in val_photos if not base.path(p['stem']).exists()]
        if missing:raise FileNotFoundError('Prepare fixed detector/refiner validation caches before segmenter training')
    for epoch in range(start,epochs+1):
        sampler.set_epoch(epoch);dataset.epoch=epoch;generator.manual_seed(c.SEED+epoch*101+rank)
        torch.cuda.reset_peak_memory_stats(device);started=time.perf_counter()
        loss=train_epoch(model,loader,optimizer,scaler,c,stage,device,accum)
        metrics={};is_best=False
        if rank==0 and not all_data and (epoch%c.VALIDATE_EVERY==0 or epoch==epochs):
            metrics=validate_epoch(module,c,cat,val_photos,pre,norm,stage,device,base,
                Path(c.OUTPUT_ROOT)/'model_selection'/f'{stage}_latest')
            is_best=metrics['pq']>best_pq
            if is_best:best_pq=metrics['pq']
        if world>1:dist.barrier(device_ids=[local])
        lr=optimizer.param_groups[0]['lr'];scheduler.step()
        local_rng=rng_state(device);all_rng=[None]*world if rank==0 else None
        if world>1:dist.gather_object(local_rng,all_rng,dst=0)
        else:all_rng=[local_rng]
        if rank==0:
            row={'epoch':epoch,'loss':loss,'lr':lr,'minutes':(time.perf_counter()-started)/60,
                 'peak_allocated_gb_rank0':torch.cuda.max_memory_allocated(device)/2**30,
                 'train_records':len(dataset),'distributed_padding_records':sampler.total_size-len(dataset),**metrics}
            history.append(row)
            state={'version':'1.0.0','stage':stage,'architecture':architecture(c,stage),
                'training_signature':signature,'split_hash':manifest['split_hash'],'annotation_hash':cat.hash,
                'normalization':norm,'epoch':epoch,'model':module.state_dict(),'optimizer':optimizer.state_dict(),
                'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),'rng':all_rng,
                'best_pq':best_pq,'history':history,'all_data':all_data,
                'trained_photo_ids':[p['stem'] for p in train_photos],
                'selection':'all-data fixed epoch; not eligible for validation' if all_data else 'native official all-annotator PQ'}
            atomic_torch(last,state)
            if is_best or all_data:atomic_torch(best,state)
            pd.DataFrame(history).to_csv(Path(c.OUTPUT_ROOT)/(('all_' if all_data else '')+stage+'_history.csv'),index=False)
            print(f'{stage} epoch {epoch}/{epochs}: loss={loss:.4f}, PQ={metrics.get("pq",float("nan")):.6f}, {row["minutes"]:.1f}m, GPU0peak={row["peak_allocated_gb_rank0"]:.2f}GB',flush=True)
        if world>1:dist.barrier(device_ids=[local])
    if dist.is_initialized():dist.destroy_process_group()


def main():
    parser=argparse.ArgumentParser();parser.add_argument('--config',required=True)
    parser.add_argument('--stage',choices=['detector','refiner','tile'],required=True)
    parser.add_argument('--all-data',action='store_true');parser.add_argument('--epochs',type=int,default=None)
    args=parser.parse_args();train(Config.load(args.config),args.stage,args.all_data,args.epochs)


if __name__=='__main__':main()
''', encoding='utf-8')


(PACKAGE_ROOT / 'NOTICE.md').write_text(r'''# Source acknowledgments

The Mask R-CNN v2 architecture, anchors, ROI mask pooling, SGD/DDP/AMP training design, ResNet18 U-Net decoder and crop geometry are based on Nomannic19's Apache-2.0 Kaggle notebook, supplied locally as `cv-solar-filament-mask-r-cnn-u-net-refiner.ipynb` (35 cells). `reference_models.py` retains the original architecture definitions. V1 changes the data handling, preprocessing, refiner input, losses, checkpoint selection, postprocessing, calibration and orchestration.

Original: https://www.kaggle.com/code/nomannic19/cv-solar-filament-mask-r-cnn-u-net-refiner

`disk.py` is adapted from ameyypawar/filament-seg, commit 81a739bb7786a44b989f5d76afbb6f9d22b5d1dd, under the MIT license. The original MIT license is included as GITHUB_REFERENCE_LICENSE. The V1 radial profile uses edge-padded smoothing to avoid convolution boundary artifacts.

Reference: https://github.com/ameyypawar/filament-seg

The official-compatible scorer preserves the all-pairs, strict IoU > 0.5 and pooled annotator semantics of the host Self_Evaluation_Notebook, version 6. Tests compare its counts and PQ with functions extracted from that public source.

Official: https://www.kaggle.com/code/azimahmadzadeh/self-evaluation-notebook

BCE + Tversky and crop-level segmentation are informed by the locally supplied MAGFiLO notebook. Its heavy YOLO architecture is not used. Artem Shelamanov's public portfolio informs the ensemble/quality interface only; its unpublished code is not claimed to have been reproduced.

https://www.shelamanov.com/portfolio/solar
''', encoding='utf-8')


(PACKAGE_ROOT / 'GITHUB_REFERENCE_LICENSE').write_text(r'''MIT License

Copyright (c) 2026 Amey Pawar

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

''', encoding='utf-8')


(PACKAGE_ROOT / 'APACHE_REFERENCE_LICENSE').write_text(r'''
                                 Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial ownership of such entity.

      "You" (or "Your") shall mean an individual or Legal Entity
      exercising permissions granted by this License.

      "Source" form shall mean the preferred form for making modifications,
      including but not limited to software source code, documentation
      source, and configuration files.

      "Object" form shall mean any form resulting from mechanical
      transformation or translation of a Source form, including but
      not limited to compiled object code, generated documentation,
      and conversions to other media types.

      "Work" shall mean the work of authorship, whether in Source or
      Object form, made available under the License, as indicated by a
      copyright notice that is included in or attached to the work
      (an example is provided in the Appendix below).

      "Derivative Works" shall mean any work, whether in Source or Object
      form, that is based on (or derived from) the Work and for which the
      editorial revisions, annotations, elaborations, or other modifications
      represent, as a whole, an original work of authorship. For the purposes
      of this License, Derivative Works shall not include works that remain
      separable from, or merely link (or bind by name) to the interfaces of,
      the Work and Derivative Works thereof.

      "Contribution" shall mean any work of authorship, including
      the original version of the Work and any modifications or additions
      to that Work or Derivative Works thereof, that is intentionally
      submitted to Licensor for inclusion in the Work by the copyright owner
      or by an individual or Legal Entity authorized to submit on behalf of
      the copyright owner. For the purposes of this definition, "submitted"
      means any form of electronic, verbal, or written communication sent
      to the Licensor or its representatives, including but not limited to
      communication on electronic mailing lists, source code control systems,
      and issue tracking systems that are managed by, or on behalf of, the
      Licensor for the purpose of discussing and improving the Work, but
      excluding communication that is conspicuously marked or otherwise
      designated in writing by the copyright owner as "Not a Contribution."

      "Contributor" shall mean Licensor and any individual or Legal Entity
      on behalf of whom a Contribution has been received by Licensor and
      subsequently incorporated within the Work.

   2. Grant of Copyright License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      copyright license to reproduce, prepare Derivative Works of,
      publicly display, publicly perform, sublicense, and distribute the
      Work and such Derivative Works in Source or Object form.

   3. Grant of Patent License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      (except as stated in this section) patent license to make, have made,
      use, offer to sell, sell, import, and otherwise transfer the Work,
      where such license applies only to those patent claims licensable
      by such Contributor that are necessarily infringed by their
      Contribution(s) alone or by combination of their Contribution(s)
      with the Work to which such Contribution(s) was submitted. If You
      institute patent litigation against any entity (including a
      cross-claim or counterclaim in a lawsuit) alleging that the Work
      or a Contribution incorporated within the Work constitutes direct
      or contributory patent infringement, then any patent licenses
      granted to You under this License for that Work shall terminate
      as of the date such litigation is filed.

   4. Redistribution. You may reproduce and distribute copies of the
      Work or Derivative Works thereof in any medium, with or without
      modifications, and in Source or Object form, provided that You
      meet the following conditions:

      (a) You must give any other recipients of the Work or
          Derivative Works a copy of this License; and

      (b) You must cause any modified files to carry prominent notices
          stating that You changed the files; and

      (c) You must retain, in the Source form of any Derivative Works
          that You distribute, all copyright, patent, trademark, and
          attribution notices from the Source form of the Work,
          excluding those notices that do not pertain to any part of
          the Derivative Works; and

      (d) If the Work includes a "NOTICE" text file as part of its
          distribution, then any Derivative Works that You distribute must
          include a readable copy of the attribution notices contained
          within such NOTICE file, excluding those notices that do not
          pertain to any part of the Derivative Works, in at least one
          of the following places: within a NOTICE text file distributed
          as part of the Derivative Works; within the Source form or
          documentation, if provided along with the Derivative Works; or,
          within a display generated by the Derivative Works, if and
          wherever such third-party notices normally appear. The contents
          of the NOTICE file are for informational purposes only and
          do not modify the License. You may add Your own attribution
          notices within Derivative Works that You distribute, alongside
          or as an addendum to the NOTICE text from the Work, provided
          that such additional attribution notices cannot be construed
          as modifying the License.

      You may add Your own copyright statement to Your modifications and
      may provide additional or different license terms and conditions
      for use, reproduction, or distribution of Your modifications, or
      for any such Derivative Works as a whole, provided Your use,
      reproduction, and distribution of the Work otherwise complies with
      the conditions stated in this License.

   5. Submission of Contributions. Unless You explicitly state otherwise,
      any Contribution intentionally submitted for inclusion in the Work
      by You to the Licensor shall be under the terms and conditions of
      this License, without any additional terms or conditions.
      Notwithstanding the above, nothing herein shall supersede or modify
      the terms of any separate license agreement you may have executed
      with Licensor regarding such Contributions.

   6. Trademarks. This License does not grant permission to use the trade
      names, trademarks, service marks, or product names of the Licensor,
      except as required for reasonable and customary use in describing the
      origin of the Work and reproducing the content of the NOTICE file.

   7. Disclaimer of Warranty. Unless required by applicable law or
      agreed to in writing, Licensor provides the Work (and each
      Contributor provides its Contributions) on an "AS IS" BASIS,
      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
      implied, including, without limitation, any warranties or conditions
      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
      PARTICULAR PURPOSE. You are solely responsible for determining the
      appropriateness of using or redistributing the Work and assume any
      risks associated with Your exercise of permissions under this License.

   8. Limitation of Liability. In no event and under no legal theory,
      whether in tort (including negligence), contract, or otherwise,
      unless required by applicable law (such as deliberate and grossly
      negligent acts) or agreed to in writing, shall any Contributor be
      liable to You for damages, including any direct, indirect, special,
      incidental, or consequential damages of any character arising as a
      result of this License or out of the use or inability to use the
      Work (including but not limited to damages for loss of goodwill,
      work stoppage, computer failure or malfunction, or any and all
      other commercial damages or losses), even if such Contributor
      has been advised of the possibility of such damages.

   9. Accepting Warranty or Additional Liability. While redistributing
      the Work or Derivative Works thereof, You may choose to offer,
      and charge a fee for, acceptance of support, warranty, indemnity,
      or other liability obligations and/or rights consistent with this
      License. However, in accepting such obligations, You may act only
      on Your own behalf and on Your sole responsibility, not on behalf
      of any other Contributor, and only if You agree to indemnify,
      defend, and hold each Contributor harmless for any liability
      incurred by, or claims asserted against, such Contributor by reason
      of your accepting any such warranty or additional liability.

   END OF TERMS AND CONDITIONS

   APPENDIX: How to apply the Apache License to your work.

      To apply the Apache License to your work, attach the following
      boilerplate notice, with the fields enclosed by brackets "[]"
      replaced with your own identifying information. (Don't include
      the brackets!)  The text should be enclosed in the appropriate
      comment syntax for the file format. We also recommend that a
      file or class name and description of purpose be included on the
      same "printed page" as the copyright notice for easier
      identification within third-party archives.

   Copyright [yyyy] [name of copyright owner]

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.
''', encoding='utf-8')


CONFIG_PATH=PACKAGE_ROOT/'config.json'
CONFIG_PATH.write_text(json.dumps(CONFIG,indent=2),encoding='utf-8')
env=os.environ.copy()
env['PYTHONPATH']=str(PACKAGE_ROOT)+os.pathsep+env.get('PYTHONPATH','')
env['OMP_NUM_THREADS']='2'
print('Starting V1.0:',CONFIG['RUN_MODE'],'| Tile branch:',CONFIG['ENABLE_TILE_BRANCH'],flush=True)
subprocess.run([sys.executable,'-m','solar_v1.driver','--config',str(CONFIG_PATH)],check=True,env=env)
if CONFIG['RUN_MODE'] in {'train_calibrate_submit','predict'}:
    assert Path('/kaggle/working/submission.csv').is_file()
    print('Submission ready: /kaggle/working/submission.csv')
